# <font color='#cec748'>This notebook is to get the individual filters for all the galaxies</font>
The notebook uses data from the get_spec_example file to:
- w

## <font color='#e55730' size=3 >Imports</font>

In [1]:
from dustmaps.config import config
config.reset()

In [2]:
from hetdex_api.detections import Detections
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.coordinates import match_coordinates_sky
from astropy.convolution import convolve, Gaussian1DKernel, Box1DKernel
import tables
import numpy as np
import os.path as op
import math
import os
import h5py

from elixer import global_config as G
G.GLOBAL_LOGGING = True
from elixer import spectrum_utilities as ESU
from elixer import catalogs


from hetdex_api.elixer_widget_cls import ElixerWidget
from hetdex_api.config import HDRconfig
from hetdex_tools.get_spec import get_spectra
from hetdex_api.shot import *
from hetdex_api.survey import FiberIndex
from hetdex_api.detections import Detections

from multiprocessing import Pool
from astropy.table import QTable, Table, Column
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from scipy import interpolate

import inspect
import scipy.stats as stats
from astropy.table import Table
from astropy.table import QTable, Table, Column
import warnings
from astropy.modeling import models
from specutils.spectra import Spectrum1D, SpectralRegion
from specutils.fitting import fit_generic_continuum
from specutils.fitting.continuum import fit_continuum
from astropy.stats import biweight_location
from scipy.integrate import quad as quad
from numpy.polynomial.polynomial import polyfit
from numpy.polynomial.polynomial import polyval
from photutils.centroids import centroid_2dg, centroid_sources
from photutils.datasets import make_4gaussians_image
from numpy import inf
from photutils.datasets import make_4gaussians_image
from photutils.centroids import centroid_quadratic

Populating dustmaps config with /home/jovyan/Hobby-Eberly-Telesco/hdr3/calib/dustmaps


In [3]:
import importlib.util
import sys

# For illustrative purposes.
name = 'lmfit'

if name in sys.modules:
    print(f"{name!r} already in sys.modules")
elif (spec := importlib.util.find_spec(name)) is not None:
    # If you choose to perform the actual import ...
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    print(f"{name!r} has been imported")
else:
    print(f"Nope Installing Now")
    !pip install lmfit
    condition = True

'lmfit' has been imported


## <font color='#e55730' size=3 >Plotting imports and settings</font>

In [4]:
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rc
import matplotlib.ticker
import matplotlib.patches as patches

%matplotlib ipympl
plt.style.use('default')
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams.update({'font.size': 14})
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['mathtext.default'] = 'regular'
plt.rcParams['xtick.direction']= 'in'
plt.rcParams['ytick.direction']= 'in'
plt.rcParams['xtick.labelsize']= 14.0
plt.rcParams['ytick.labelsize']= 14.0

## <font color='#e55730' size=3 >Values</font>

In [5]:
c = 299792.5 #km s^-1
H_0 = 70 #km s^-1 MpC^-1
R = 800

V = c/R
print(V)

374.740625


## <font color='#e55730' size=3 >Definitions</font>

In [6]:
pwd

'/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog'

## <font color='#e55730' size=3 >Creating Files</font>

In [7]:
File_New_Catalog = "/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/hetdex_agn_hdr4.fits"


In [8]:
Data_New_Catalog = Table.read(File_New_Catalog, format = 'fits')

print(len(Data_New_Catalog["detectid_best"]))


15940


In [9]:
catlib = catalogs.CatalogLibrary()


## <font color='#e55730' size=3 >Getting the Redshift and AGN confirmed</font>

In [14]:
sel = (np.array(Data_New_Catalog['z'] > 0.25) & np.array(Data_New_Catalog['z'] < 0.96)) #This is the redshift range to capture MgII
sel = sel & np.array(Data_New_Catalog['zflag'] == 1) #We only want zflag = 1 because it means the redshift has been confirmed by line pairs or SDSS
sel = sel & np.array(Data_New_Catalog["agn_flag"] == 1) #We only want agns


Data_Confirmed_Redshift_And_AGN_Status = Data_New_Catalog[sel]

print(len(Data_Confirmed_Redshift_And_AGN_Status["z"]))


1320


In [12]:
#Let's just do 6 for now
##Making the groups
z_025_035 = []
z_035_045 = []
z_045_055 = []
z_055_065 = []
z_065_075 = []
z_075_085 = []
z_085_096 = []

z_025_035_index = []
z_035_045_index = []
z_045_055_index = []
z_055_065_index = []
z_065_075_index = []
z_075_085_index = []
z_085_096_index = [] 


a = -1
for i in Data_Confirmed_Redshift_And_AGN_Status["z"]:
    a = a+1
    #print(i)
    if 0.25 <= i < 0.35:
        #print(i)
        z_025_035.append(i)
        z_025_035_index.append(a)
    if 0.35 <= i < 0.45:
        z_035_045.append(i)
        z_035_045_index.append(a)
    if 0.45 <= i < 0.55:
        z_045_055.append(i)
        z_045_055_index.append(a)
    if 0.55 <= i < 0.65:
        z_055_065.append(i)
        z_055_065_index.append(a)
    if 0.65 <= i < 0.75:
        z_065_075.append(i)
        z_065_075_index.append(a)
    if 0.75 <= i < 0.85:
        z_075_085.append(i)
        z_075_085_index.append(a)
    if 0.85<= i <= 0.96:
        z_085_096.append(i)
        z_085_096_index.append(a)
    else:
        continue

In [13]:
test_a = np.hstack((z_025_035, z_035_045, z_045_055, z_055_065, z_065_075, z_075_085, z_085_096))
print(len(test_a))

1320


In [ ]:
G_Data_Array = []
G_RA_Array = []
G_DEC_Array = []
G_Detect_ID = []
G_Redshift_Array = []
G_Redshift_Err_Array = []
G_SNR_Array = []
G_Mag_Array = []
G_Mag_Err_Array = []
G_Aperture_Array = []

R_Data_Array = []
R_RA_Array = []
R_DEC_Array = []
R_Detect_ID = []
R_Redshift_Array = []
R_Redshift_Err_Array = []
R_SNR_Array = []
R_Mag_Array = []
R_Mag_Err_Array = []
R_Aperture_Array = []

I_Data_Array = []
I_RA_Array = []
I_DEC_Array = []
I_Detect_ID = []
I_Redshift_Array = []
I_Redshift_Err_Array = []
I_SNR_Array = []
I_Mag_Array = []
I_Mag_Err_Array = []
I_Aperture_Array = []

Z_Data_Array = []
Z_RA_Array = []
Z_DEC_Array = []
Z_Detect_ID = []
Z_Redshift_Array = []
Z_Redshift_Err_Array = []
Z_SNR_Array = []
Z_Mag_Array = []
Z_Mag_Err_Array = []
Z_Aperture_Array = []

Y_Data_Array = []
Y_RA_Array = []
Y_DEC_Array = []
Y_Detect_ID = []
Y_Redshift_Array = []
Y_Redshift_Err_Array = []
Y_SNR_Array = []
Y_Mag_Array = []
Y_Mag_Err_Array = []
Y_Aperture_Array = []




for i in np.arange(len(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_025_035_index])): 

#for i in np.arange(10): 
    coord7 = SkyCoord(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_025_035_index][i], Data_Confirmed_Redshift_And_AGN_Status["DEC"][z_025_035_index][i], unit = 'deg')
    cutouts7 = catlib.get_cutouts(position=coord7,radius=7.,aperture=1.5,dynamic=False,first=False,nudge=False,filter=None)

    #print(cutouts7)
    if len(cutouts7) > 0:
        cutout = cutouts7[0] 
        #print(cutout['instrument'])
        
        if cutout['instrument'] == "HSC SSP":
            for j in cutouts7:
                #print(j['details']["filter_name"])
                
                if j['details']["filter_name"] == "g":
                    #print("g")
                    #print(j['cutout'].error)
                    G_Data_Array.append(j['cutout'].data)
                    G_RA_Array.append(j['details']['ra'])
                    G_DEC_Array.append(j['details']['dec'])
                    G_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_025_035_index][i])
                    G_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_025_035_index][i])
                    G_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_025_035_index][i])
                    G_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_025_035_index][i])
                    G_Mag_Array.append(j['details']['mag'])
                    G_Mag_Err_Array.append(j['details']['mag_err'])
                    G_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "r":
                    #print("r")
                    R_Data_Array.append(j['cutout'].data)
                    R_RA_Array.append(j['details']['ra'])
                    R_DEC_Array.append(j['details']['dec'])
                    R_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_025_035_index][i])
                    R_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_025_035_index][i])
                    R_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_025_035_index][i])
                    R_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_025_035_index][i])
                    R_Mag_Array.append(j['details']['mag'])
                    R_Mag_Err_Array.append(j['details']['mag_err'])
                    R_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "i":
                    I_Data_Array.append(j['cutout'].data)
                    I_RA_Array.append(j['details']['ra'])
                    I_DEC_Array.append(j['details']['dec'])
                    I_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_025_035_index][i])
                    I_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_025_035_index][i])
                    I_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_025_035_index][i])
                    I_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_025_035_index][i])
                    I_Mag_Array.append(j['details']['mag'])
                    I_Mag_Err_Array.append(j['details']['mag_err'])
                    I_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "z":
                    Z_Data_Array.append(j['cutout'].data)
                    Z_RA_Array.append(j['details']['ra'])
                    Z_DEC_Array.append(j['details']['dec'])
                    Z_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_025_035_index][i])
                    Z_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_025_035_index][i])
                    Z_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_025_035_index][i])
                    Z_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_025_035_index][i])
                    Z_Mag_Array.append(j['details']['mag'])
                    Z_Mag_Err_Array.append(j['details']['mag_err'])
                    Z_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "y":
                    Y_Data_Array.append(j['cutout'].data)
                    Y_RA_Array.append(j['details']['ra'])
                    Y_DEC_Array.append(j['details']['dec'])
                    Y_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_025_035_index][i])
                    Y_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_025_035_index][i])
                    Y_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_025_035_index][i])
                    Y_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_025_035_index][i])
                    Y_Mag_Array.append(j['details']['mag'])
                    Y_Mag_Err_Array.append(j['details']['mag_err'])
                    Y_Aperture_Array.append(j['aperture'])


In [ ]:
print(j['cutout'].__dict__)

In [ ]:
print(cutouts7)

In [ ]:
Table_start_a = ([(G_Data_Array[0])])
Table_start_b = ([(G_RA_Array[0])])
Table_start_c = ([(G_DEC_Array[0])])
Table_start_d = ([(G_Detect_ID[0])])
Table_start_e = ([(G_Redshift_Array[0])])
Table_start_f = ([(G_Mag_Array[0])])
Table_start_g = ([(G_Mag_Err_Array[0])])
Table_start_h = ([(G_Aperture_Array[0])])
Table_start_i = ([(G_Redshift_Err_Array[0])])
Table_start_j = ([(G_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("G_Data","G_RA","G_DEC", "G_Detect_ID", "G_Z", "G_Mag", "G_Mag_Err", "G_Aperture_Array", "G_Z_Err", "G_MgII_SNR"))


print(len(G_Data_Array), len(R_Data_Array), len(I_Data_Array), len(Z_Data_Array), len(Y_Data_Array))

In [ ]:
for i in np.arange(1,len(G_RA_Array),1):
    a = ([(G_Data_Array[i])])
    b = ([(G_RA_Array[i])])
    c = ([(G_DEC_Array[i])])
    d = ([(G_Detect_ID[i])])
    e = ([(G_Redshift_Array[i])])
    f = ([(G_Mag_Array[i])])
    g = ([(G_Mag_Err_Array[i])])
    h = ([(G_Aperture_Array[i])])
    ii = ([(G_Redshift_Err_Array[i])])
    j = ([(G_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_025_035_G.fits", overwrite = True)

In [ ]:
Table_start_a = ([(R_Data_Array[0])])
Table_start_b = ([(R_RA_Array[0])])
Table_start_c = ([(R_DEC_Array[0])])
Table_start_d = ([(R_Detect_ID[0])])
Table_start_e = ([(R_Redshift_Array[0])])
Table_start_f = ([(R_Mag_Array[0])])
Table_start_g = ([(R_Mag_Err_Array[0])])
Table_start_h = ([(R_Aperture_Array[0])])
Table_start_i = ([(R_Redshift_Err_Array[0])])
Table_start_j = ([(R_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("R_Data","R_RA","R_DEC", "R_Detect_ID", "R_Z", "R_Mag", "R_Mag_Err", "R_Aperture_Array", "R_Z_Err", "R_MgII_SNR"))


In [ ]:
print(R_Mag_Array)
print("")
print(R_Mag_Err_Array)

In [ ]:
for i in np.arange(1,len(R_RA_Array),1):
    a = ([(R_Data_Array[i])])
    b = ([(R_RA_Array[i])])
    c = ([(R_DEC_Array[i])])
    d = ([(R_Detect_ID[i])])
    e = ([(R_Redshift_Array[i])])
    f = ([(R_Mag_Array[i])])
    g = ([(R_Mag_Err_Array[i])])
    h = ([(R_Aperture_Array[i])])
    ii = ([(R_Redshift_Err_Array[i])])
    j = ([(R_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_025_035_R.fits", overwrite = True)

In [ ]:
Table_start_a = ([(I_Data_Array[0])])
Table_start_b = ([(I_RA_Array[0])])
Table_start_c = ([(I_DEC_Array[0])])
Table_start_d = ([(I_Detect_ID[0])])
Table_start_e = ([(I_Redshift_Array[0])])
Table_start_f = ([(I_Mag_Array[0])])
Table_start_g = ([(I_Mag_Err_Array[0])])
Table_start_h = ([(I_Aperture_Array[0])])
Table_start_i = ([(I_Redshift_Err_Array[0])])
Table_start_j = ([(I_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("I_Data","I_RA","I_DEC", "I_Detect_ID", "I_Z", "I_Mag", "I_Mag_Err", "I_Aperture_Array", "I_Z_Err", "I_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(I_RA_Array),1):
    a = ([(I_Data_Array[i])])
    b = ([(I_RA_Array[i])])
    c = ([(I_DEC_Array[i])])
    d = ([(I_Detect_ID[i])])
    e = ([(I_Redshift_Array[i])])
    f = ([(I_Mag_Array[i])])
    g = ([(I_Mag_Err_Array[i])])
    h = ([(I_Aperture_Array[i])])
    ii = ([(I_Redshift_Err_Array[i])])
    j = ([(I_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_025_035_I.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Z_Data_Array[0])])
Table_start_b = ([(Z_RA_Array[0])])
Table_start_c = ([(Z_DEC_Array[0])])
Table_start_d = ([(Z_Detect_ID[0])])
Table_start_e = ([(Z_Redshift_Array[0])])
Table_start_f = ([(Z_Mag_Array[0])])
Table_start_g = ([(Z_Mag_Err_Array[0])])
Table_start_h = ([(Z_Aperture_Array[0])])
Table_start_i = ([(Z_Redshift_Err_Array[0])])
Table_start_j = ([(Z_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Z_Data","Z_RA","Z_DEC", "Z_Detect_ID", "Z_Z", "Z_Mag", "Z_Mag_Err", "Z_Aperture_Array", "Z_Z_Err", "Z_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Z_RA_Array),1):
    a = ([(Z_Data_Array[i])])
    b = ([(Z_RA_Array[i])])
    c = ([(Z_DEC_Array[i])])
    d = ([(Z_Detect_ID[i])])
    e = ([(Z_Redshift_Array[i])])
    f = ([(Z_Mag_Array[i])])
    g = ([(Z_Mag_Err_Array[i])])
    h = ([(Z_Aperture_Array[i])])
    ii = ([(Z_Redshift_Err_Array[i])])
    j = ([(Z_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_025_035_Z.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Y_Data_Array[0])])
Table_start_b = ([(Y_RA_Array[0])])
Table_start_c = ([(Y_DEC_Array[0])])
Table_start_d = ([(Y_Detect_ID[0])])
Table_start_e = ([(Y_Redshift_Array[0])])
Table_start_f = ([(Y_Mag_Array[0])])
Table_start_g = ([(Y_Mag_Err_Array[0])])
Table_start_h = ([(Y_Aperture_Array[0])])
Table_start_i = ([(Y_Redshift_Err_Array[0])])
Table_start_j = ([(Y_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Y_Data","Y_RA","Y_DEC", "Y_Detect_ID", "Y_Z", "Y_Mag", "Y_Mag_Err", "Y_Aperture_Array", "Y_Z_Err", "Y_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_025_035_Y.fits", overwrite = True)

<hr style="border:4px solid blue">

In [ ]:
G_Data_Array = []
G_RA_Array = []
G_DEC_Array = []
G_Detect_ID = []
G_Redshift_Array = []
G_Redshift_Err_Array = []
G_SNR_Array = []
G_Mag_Array = []
G_Mag_Err_Array = []
G_Aperture_Array = []

R_Data_Array = []
R_RA_Array = []
R_DEC_Array = []
R_Detect_ID = []
R_Redshift_Array = []
R_Redshift_Err_Array = []
R_SNR_Array = []
R_Mag_Array = []
R_Mag_Err_Array = []
R_Aperture_Array = []

I_Data_Array = []
I_RA_Array = []
I_DEC_Array = []
I_Detect_ID = []
I_Redshift_Array = []
I_Redshift_Err_Array = []
I_SNR_Array = []
I_Mag_Array = []
I_Mag_Err_Array = []
I_Aperture_Array = []

Z_Data_Array = []
Z_RA_Array = []
Z_DEC_Array = []
Z_Detect_ID = []
Z_Redshift_Array = []
Z_Redshift_Err_Array = []
Z_SNR_Array = []
Z_Mag_Array = []
Z_Mag_Err_Array = []
Z_Aperture_Array = []

Y_Data_Array = []
Y_RA_Array = []
Y_DEC_Array = []
Y_Detect_ID = []
Y_Redshift_Array = []
Y_Redshift_Err_Array = []
Y_SNR_Array = []
Y_Mag_Array = []
Y_Mag_Err_Array = []
Y_Aperture_Array = []



for i in np.arange(len(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_035_045_index])): 

#for i in np.arange(10): 
    coord7 = SkyCoord(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_035_045_index][i], Data_Confirmed_Redshift_And_AGN_Status["DEC"][z_035_045_index][i], unit = 'deg')
    cutouts7 = catlib.get_cutouts(position=coord7,radius=7.,aperture=1.5,dynamic=False,first=False,nudge=False,filter=None)

    #print(cutouts7)
    if len(cutouts7) > 0:
        cutout = cutouts7[0] 
        #print(cutout['instrument'])
        
        if cutout['instrument'] == "HSC SSP":
            for j in cutouts7:
                #print(j['details']["filter_name"])
                
                if j['details']["filter_name"] == "g":
                    #print("g")
                    #print(j['cutout'].error)
                    G_Data_Array.append(j['cutout'].data)
                    G_RA_Array.append(j['details']['ra'])
                    G_DEC_Array.append(j['details']['dec'])
                    G_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_035_045_index][i])
                    G_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_035_045_index][i])
                    G_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_035_045_index][i])
                    G_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_035_045_index][i])
                    G_Mag_Array.append(j['details']['mag'])
                    G_Mag_Err_Array.append(j['details']['mag_err'])
                    G_Aperture_Array.append(j['aperture'])


                elif j['details']["filter_name"] == "r":
                    #print("r")
                    R_Data_Array.append(j['cutout'].data)
                    R_RA_Array.append(j['details']['ra'])
                    R_DEC_Array.append(j['details']['dec'])
                    R_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_035_045_index][i])
                    R_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_035_045_index][i])
                    R_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_035_045_index][i])
                    R_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_035_045_index][i])
                    R_Mag_Array.append(j['details']['mag'])
                    R_Mag_Err_Array.append(j['details']['mag_err'])
                    R_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "i":
                    I_Data_Array.append(j['cutout'].data)
                    I_RA_Array.append(j['details']['ra'])
                    I_DEC_Array.append(j['details']['dec'])
                    I_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_035_045_index][i])
                    I_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_035_045_index][i])
                    I_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_035_045_index][i])
                    I_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_035_045_index][i])
                    I_Mag_Array.append(j['details']['mag'])
                    I_Mag_Err_Array.append(j['details']['mag_err'])
                    I_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "z":
                    Z_Data_Array.append(j['cutout'].data)
                    Z_RA_Array.append(j['details']['ra'])
                    Z_DEC_Array.append(j['details']['dec'])
                    Z_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_035_045_index][i])
                    Z_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_035_045_index][i])
                    Z_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_035_045_index][i])
                    Z_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_035_045_index][i])
                    Z_Mag_Array.append(j['details']['mag'])
                    Z_Mag_Err_Array.append(j['details']['mag_err'])
                    Z_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "y":
                    Y_Data_Array.append(j['cutout'].data)
                    Y_RA_Array.append(j['details']['ra'])
                    Y_DEC_Array.append(j['details']['dec'])
                    Y_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_035_045_index][i])
                    Y_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_035_045_index][i])
                    Y_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_035_045_index][i])
                    Y_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_035_045_index][i])
                    Y_Mag_Array.append(j['details']['mag'])
                    Y_Mag_Err_Array.append(j['details']['mag_err'])
                    Y_Aperture_Array.append(j['aperture'])


In [ ]:
Table_start_a = ([(G_Data_Array[0])])
Table_start_b = ([(G_RA_Array[0])])
Table_start_c = ([(G_DEC_Array[0])])
Table_start_d = ([(G_Detect_ID[0])])
Table_start_e = ([(G_Redshift_Array[0])])
Table_start_f = ([(G_Mag_Array[0])])
Table_start_g = ([(G_Mag_Err_Array[0])])
Table_start_h = ([(G_Aperture_Array[0])])
Table_start_i = ([(G_Redshift_Err_Array[0])])
Table_start_j = ([(G_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("G_Data","G_RA","G_DEC", "G_Detect_ID", "G_Z", "G_Mag", "G_Mag_Err", "G_Aperture_Array", "G_Z_Err", "G_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(G_RA_Array),1):
    a = ([(G_Data_Array[i])])
    b = ([(G_RA_Array[i])])
    c = ([(G_DEC_Array[i])])
    d = ([(G_Detect_ID[i])])
    e = ([(G_Redshift_Array[i])])
    f = ([(G_Mag_Array[i])])
    g = ([(G_Mag_Err_Array[i])])
    h = ([(G_Aperture_Array[i])])
    ii = ([(G_Redshift_Err_Array[i])])
    j = ([(G_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_035_045_G.fits", overwrite = True)

In [ ]:
Table_start_a = ([(R_Data_Array[0])])
Table_start_b = ([(R_RA_Array[0])])
Table_start_c = ([(R_DEC_Array[0])])
Table_start_d = ([(R_Detect_ID[0])])
Table_start_e = ([(R_Redshift_Array[0])])
Table_start_f = ([(R_Mag_Array[0])])
Table_start_g = ([(R_Mag_Err_Array[0])])
Table_start_h = ([(R_Aperture_Array[0])])
Table_start_i = ([(R_Redshift_Err_Array[0])])
Table_start_j = ([(R_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("R_Data","R_RA","R_DEC", "R_Detect_ID", "R_Z", "R_Mag", "R_Mag_Err", "R_Aperture_Array", "R_Z_Err", "R_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(R_RA_Array),1):
    a = ([(R_Data_Array[i])])
    b = ([(R_RA_Array[i])])
    c = ([(R_DEC_Array[i])])
    d = ([(R_Detect_ID[i])])
    e = ([(R_Redshift_Array[i])])
    f = ([(R_Mag_Array[i])])
    g = ([(R_Mag_Err_Array[i])])
    h = ([(R_Aperture_Array[i])])
    ii = ([(R_Redshift_Err_Array[i])])
    j = ([(R_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_035_045_R.fits", overwrite = True)

In [ ]:
Table_start_a = ([(I_Data_Array[0])])
Table_start_b = ([(I_RA_Array[0])])
Table_start_c = ([(I_DEC_Array[0])])
Table_start_d = ([(I_Detect_ID[0])])
Table_start_e = ([(I_Redshift_Array[0])])
Table_start_f = ([(I_Mag_Array[0])])
Table_start_g = ([(I_Mag_Err_Array[0])])
Table_start_h = ([(I_Aperture_Array[0])])
Table_start_i = ([(I_Redshift_Err_Array[0])])
Table_start_j = ([(I_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("I_Data","I_RA","I_DEC", "I_Detect_ID", "I_Z", "I_Mag", "I_Mag_Err", "I_Aperture_Array", "I_Z_Err", "I_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(I_RA_Array),1):
    a = ([(I_Data_Array[i])])
    b = ([(I_RA_Array[i])])
    c = ([(I_DEC_Array[i])])
    d = ([(I_Detect_ID[i])])
    e = ([(I_Redshift_Array[i])])
    f = ([(I_Mag_Array[i])])
    g = ([(I_Mag_Err_Array[i])])
    h = ([(I_Aperture_Array[i])])
    ii = ([(I_Redshift_Err_Array[i])])
    j = ([(I_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_035_045_I.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Z_Data_Array[0])])
Table_start_b = ([(Z_RA_Array[0])])
Table_start_c = ([(Z_DEC_Array[0])])
Table_start_d = ([(Z_Detect_ID[0])])
Table_start_e = ([(Z_Redshift_Array[0])])
Table_start_f = ([(Z_Mag_Array[0])])
Table_start_g = ([(Z_Mag_Err_Array[0])])
Table_start_h = ([(Z_Aperture_Array[0])])
Table_start_i = ([(Z_Redshift_Err_Array[0])])
Table_start_j = ([(Z_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Z_Data","Z_RA","Z_DEC", "Z_Detect_ID", "Z_Z", "Z_Mag", "Z_Mag_Err", "Z_Aperture_Array", "Z_Z_Err", "Z_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Z_RA_Array),1):
    a = ([(Z_Data_Array[i])])
    b = ([(Z_RA_Array[i])])
    c = ([(Z_DEC_Array[i])])
    d = ([(Z_Detect_ID[i])])
    e = ([(Z_Redshift_Array[i])])
    f = ([(Z_Mag_Array[i])])
    g = ([(Z_Mag_Err_Array[i])])
    h = ([(Z_Aperture_Array[i])])
    ii = ([(Z_Redshift_Err_Array[i])])
    j = ([(Z_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_035_045_Z.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Y_Data_Array[0])])
Table_start_b = ([(Y_RA_Array[0])])
Table_start_c = ([(Y_DEC_Array[0])])
Table_start_d = ([(Y_Detect_ID[0])])
Table_start_e = ([(Y_Redshift_Array[0])])
Table_start_f = ([(Y_Mag_Array[0])])
Table_start_g = ([(Y_Mag_Err_Array[0])])
Table_start_h = ([(Y_Aperture_Array[0])])
Table_start_i = ([(Y_Redshift_Err_Array[0])])
Table_start_j = ([(Y_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Y_Data","Y_RA","Y_DEC", "Y_Detect_ID", "Y_Z", "Y_Mag", "Y_Mag_Err", "Y_Aperture_Array", "Y_Z_Err", "Y_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_035_045_Y.fits", overwrite = True)

<hr style="border:4px solid blue">

In [ ]:
G_Data_Array = []
G_RA_Array = []
G_DEC_Array = []
G_Detect_ID = []
G_Redshift_Array = []
G_Redshift_Err_Array = []
G_SNR_Array = []
G_Mag_Array = []
G_Mag_Err_Array = []
G_Aperture_Array = []

R_Data_Array = []
R_RA_Array = []
R_DEC_Array = []
R_Detect_ID = []
R_Redshift_Array = []
R_Redshift_Err_Array = []
R_SNR_Array = []
R_Mag_Array = []
R_Mag_Err_Array = []
R_Aperture_Array = []

I_Data_Array = []
I_RA_Array = []
I_DEC_Array = []
I_Detect_ID = []
I_Redshift_Array = []
I_Redshift_Err_Array = []
I_SNR_Array = []
I_Mag_Array = []
I_Mag_Err_Array = []
I_Aperture_Array = []

Z_Data_Array = []
Z_RA_Array = []
Z_DEC_Array = []
Z_Detect_ID = []
Z_Redshift_Array = []
Z_Redshift_Err_Array = []
Z_SNR_Array = []
Z_Mag_Array = []
Z_Mag_Err_Array = []
Z_Aperture_Array = []

Y_Data_Array = []
Y_RA_Array = []
Y_DEC_Array = []
Y_Detect_ID = []
Y_Redshift_Array = []
Y_Redshift_Err_Array = []
Y_SNR_Array = []
Y_Mag_Array = []
Y_Mag_Err_Array = []
Y_Aperture_Array = []





for i in np.arange(len(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_045_055_index])): 

#for i in np.arange(10): 
    coord7 = SkyCoord(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_045_055_index][i], Data_Confirmed_Redshift_And_AGN_Status["DEC"][z_045_055_index][i], unit = 'deg')
    cutouts7 = catlib.get_cutouts(position=coord7,radius=7.,aperture=1.5,dynamic=False,first=False,nudge=False,filter=None)

    #print(cutouts7)
    if len(cutouts7) > 0:
        cutout = cutouts7[0] 
        #print(cutout['instrument'])
        
        if cutout['instrument'] == "HSC SSP":
            for j in cutouts7:
                #print(j['details']["filter_name"])
                
                if j['details']["filter_name"] == "g":
                    #print("g")
                    #print(j['cutout'].error)
                    G_Data_Array.append(j['cutout'].data)
                    G_RA_Array.append(j['details']['ra'])
                    G_DEC_Array.append(j['details']['dec'])
                    G_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_045_055_index][i])
                    G_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_045_055_index][i])
                    G_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_045_055_index][i])
                    G_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_045_055_index][i])
                    G_Mag_Array.append(j['details']['mag'])
                    G_Mag_Err_Array.append(j['details']['mag_err'])
                    G_Aperture_Array.append(j['aperture'])


                elif j['details']["filter_name"] == "r":
                    #print("r")
                    R_Data_Array.append(j['cutout'].data)
                    R_RA_Array.append(j['details']['ra'])
                    R_DEC_Array.append(j['details']['dec'])
                    R_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_045_055_index][i])
                    R_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_045_055_index][i])
                    R_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_045_055_index][i])
                    R_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_045_055_index][i])
                    R_Mag_Array.append(j['details']['mag'])
                    R_Mag_Err_Array.append(j['details']['mag_err'])
                    R_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "i":
                    I_Data_Array.append(j['cutout'].data)
                    I_RA_Array.append(j['details']['ra'])
                    I_DEC_Array.append(j['details']['dec'])
                    I_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_045_055_index][i])
                    I_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_045_055_index][i])
                    I_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_045_055_index][i])
                    I_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_045_055_index][i])
                    I_Mag_Array.append(j['details']['mag'])
                    I_Mag_Err_Array.append(j['details']['mag_err'])
                    I_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "z":
                    Z_Data_Array.append(j['cutout'].data)
                    Z_RA_Array.append(j['details']['ra'])
                    Z_DEC_Array.append(j['details']['dec'])
                    Z_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_045_055_index][i])
                    Z_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_045_055_index][i])
                    Z_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_045_055_index][i])
                    Z_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_045_055_index][i])
                    Z_Mag_Array.append(j['details']['mag'])
                    Z_Mag_Err_Array.append(j['details']['mag_err'])
                    Z_Aperture_Array.append(j['aperture'])
                    
                elif j['details']["filter_name"] == "y":
                    Y_Data_Array.append(j['cutout'].data)
                    Y_RA_Array.append(j['details']['ra'])
                    Y_DEC_Array.append(j['details']['dec'])
                    Y_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_045_055_index][i])
                    Y_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_045_055_index][i])
                    Y_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_045_055_index][i])
                    Y_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_045_055_index][i])
                    Y_Mag_Array.append(j['details']['mag'])
                    Y_Mag_Err_Array.append(j['details']['mag_err'])
                    Y_Aperture_Array.append(j['aperture'])


                    

In [ ]:
Table_start_a = ([(G_Data_Array[0])])
Table_start_b = ([(G_RA_Array[0])])
Table_start_c = ([(G_DEC_Array[0])])
Table_start_d = ([(G_Detect_ID[0])])
Table_start_e = ([(G_Redshift_Array[0])])
Table_start_f = ([(G_Mag_Array[0])])
Table_start_g = ([(G_Mag_Err_Array[0])])
Table_start_h = ([(G_Aperture_Array[0])])
Table_start_i = ([(G_Redshift_Err_Array[0])])
Table_start_j = ([(G_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("G_Data","G_RA","G_DEC", "G_Detect_ID", "G_Z", "G_Mag", "G_Mag_Err", "G_Aperture_Array", "G_Z_Err", "G_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(G_RA_Array),1):
    a = ([(G_Data_Array[i])])
    b = ([(G_RA_Array[i])])
    c = ([(G_DEC_Array[i])])
    d = ([(G_Detect_ID[i])])
    e = ([(G_Redshift_Array[i])])
    f = ([(G_Mag_Array[i])])
    g = ([(G_Mag_Err_Array[i])])
    h = ([(G_Aperture_Array[i])])
    ii = ([(G_Redshift_Err_Array[i])])
    j = ([(G_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_045_055_G.fits", overwrite = True)

In [ ]:
Table_start_a = ([(R_Data_Array[0])])
Table_start_b = ([(R_RA_Array[0])])
Table_start_c = ([(R_DEC_Array[0])])
Table_start_d = ([(R_Detect_ID[0])])
Table_start_e = ([(R_Redshift_Array[0])])
Table_start_f = ([(R_Mag_Array[0])])
Table_start_g = ([(R_Mag_Err_Array[0])])
Table_start_h = ([(R_Aperture_Array[0])])
Table_start_i = ([(R_Redshift_Err_Array[0])])
Table_start_j = ([(R_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("R_Data","R_RA","R_DEC", "R_Detect_ID", "R_Z", "R_Mag", "R_Mag_Err", "R_Aperture_Array", "R_Z_Err", "R_MgII_SNR"))


In [ ]:
print(len(G_Data_Array), len(R_Data_Array), len(I_Data_Array), len(Z_Data_Array), len(Y_Data_Array))

In [ ]:
for i in np.arange(1,len(R_RA_Array),1):
    a = ([(R_Data_Array[i])])
    b = ([(R_RA_Array[i])])
    c = ([(R_DEC_Array[i])])
    d = ([(R_Detect_ID[i])])
    e = ([(R_Redshift_Array[i])])
    f = ([(R_Mag_Array[i])])
    g = ([(R_Mag_Err_Array[i])])
    h = ([(R_Aperture_Array[i])])
    ii = ([(R_Redshift_Err_Array[i])])
    j = ([(R_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_045_055_R.fits", overwrite = True)

In [ ]:
Table_start_a = ([(I_Data_Array[0])])
Table_start_b = ([(I_RA_Array[0])])
Table_start_c = ([(I_DEC_Array[0])])
Table_start_d = ([(I_Detect_ID[0])])
Table_start_e = ([(I_Redshift_Array[0])])
Table_start_f = ([(I_Mag_Array[0])])
Table_start_g = ([(I_Mag_Err_Array[0])])
Table_start_h = ([(I_Aperture_Array[0])])
Table_start_i = ([(I_Redshift_Err_Array[0])])
Table_start_j = ([(I_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("I_Data","I_RA","I_DEC", "I_Detect_ID", "I_Z", "I_Mag", "I_Mag_Err", "I_Aperture_Array", "I_Z_Err", "I_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(I_RA_Array),1):
    a = ([(I_Data_Array[i])])
    b = ([(I_RA_Array[i])])
    c = ([(I_DEC_Array[i])])
    d = ([(I_Detect_ID[i])])
    e = ([(I_Redshift_Array[i])])
    f = ([(I_Mag_Array[i])])
    g = ([(I_Mag_Err_Array[i])])
    h = ([(I_Aperture_Array[i])])
    ii = ([(I_Redshift_Err_Array[i])])
    j = ([(I_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_045_055_I.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Z_Data_Array[0])])
Table_start_b = ([(Z_RA_Array[0])])
Table_start_c = ([(Z_DEC_Array[0])])
Table_start_d = ([(Z_Detect_ID[0])])
Table_start_e = ([(Z_Redshift_Array[0])])
Table_start_f = ([(Z_Mag_Array[0])])
Table_start_g = ([(Z_Mag_Err_Array[0])])
Table_start_h = ([(Z_Aperture_Array[0])])
Table_start_i = ([(Z_Redshift_Err_Array[0])])
Table_start_j = ([(Z_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Z_Data","Z_RA","Z_DEC", "Z_Detect_ID", "Z_Z", "Z_Mag", "Z_Mag_Err", "Z_Aperture_Array", "Z_Z_Err", "Z_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Z_RA_Array),1):
    a = ([(Z_Data_Array[i])])
    b = ([(Z_RA_Array[i])])
    c = ([(Z_DEC_Array[i])])
    d = ([(Z_Detect_ID[i])])
    e = ([(Z_Redshift_Array[i])])
    f = ([(Z_Mag_Array[i])])
    g = ([(Z_Mag_Err_Array[i])])
    h = ([(Z_Aperture_Array[i])])
    ii = ([(Z_Redshift_Err_Array[i])])
    j = ([(Z_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_045_055_Z.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Y_Data_Array[0])])
Table_start_b = ([(Y_RA_Array[0])])
Table_start_c = ([(Y_DEC_Array[0])])
Table_start_d = ([(Y_Detect_ID[0])])
Table_start_e = ([(Y_Redshift_Array[0])])
Table_start_f = ([(Y_Mag_Array[0])])
Table_start_g = ([(Y_Mag_Err_Array[0])])
Table_start_h = ([(Y_Aperture_Array[0])])
Table_start_i = ([(Y_Redshift_Err_Array[0])])
Table_start_j = ([(Y_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Y_Data","Y_RA","Y_DEC", "Y_Detect_ID", "Y_Z", "Y_Mag", "Y_Mag_Err", "Y_Aperture_Array", "Y_Z_Err", "Y_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_045_055_Y.fits", overwrite = True)

<hr style="border:4px solid blue">

In [ ]:
G_Data_Array = []
G_RA_Array = []
G_DEC_Array = []
G_Detect_ID = []
G_Redshift_Array = []
G_Redshift_Err_Array = []
G_SNR_Array = []
G_Mag_Array = []
G_Mag_Err_Array = []
G_Aperture_Array = []


R_Data_Array = []
R_RA_Array = []
R_DEC_Array = []
R_Detect_ID = []
R_Redshift_Array = []
R_Redshift_Err_Array = []
R_SNR_Array = []
R_Mag_Array = []
R_Mag_Err_Array = []
R_Aperture_Array = []


I_Data_Array = []
I_RA_Array = []
I_DEC_Array = []
I_Detect_ID = []
I_Redshift_Array = []
I_Redshift_Err_Array = []
I_SNR_Array = []
I_Mag_Array = []
I_Mag_Err_Array = []
I_Aperture_Array = []

Z_Data_Array = []
Z_RA_Array = []
Z_DEC_Array = []
Z_Detect_ID = []
Z_Redshift_Array = []
Z_Redshift_Err_Array = []
Z_SNR_Array = []
Z_Mag_Array = []
Z_Mag_Err_Array = []
Z_Aperture_Array = []

Y_Data_Array = []
Y_RA_Array = []
Y_DEC_Array = []
Y_Detect_ID = []
Y_Redshift_Array = []
Y_Redshift_Err_Array = []
Y_SNR_Array = []
Y_Mag_Array = []
Y_Mag_Err_Array = []
Y_Aperture_Array = []





for i in np.arange(len(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_055_065_index])): 

#for i in np.arange(10): 
    coord7 = SkyCoord(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_055_065_index][i], Data_Confirmed_Redshift_And_AGN_Status["DEC"][z_055_065_index][i], unit = 'deg')
    cutouts7 = catlib.get_cutouts(position=coord7,radius=7.,aperture=1.5,dynamic=False,first=False,nudge=False,filter=None)

    #print(cutouts7)
    if len(cutouts7) > 0:
        cutout = cutouts7[0] 
        #print(cutout['instrument'])
        
        if cutout['instrument'] == "HSC SSP":
            for j in cutouts7:
                #print(j['details']["filter_name"])
                
                if j['details']["filter_name"] == "g":
                    #print("g")
                    #print(j['cutout'].error)
                    G_Data_Array.append(j['cutout'].data)
                    G_RA_Array.append(j['details']['ra'])
                    G_DEC_Array.append(j['details']['dec'])
                    G_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_055_065_index][i])
                    G_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_055_065_index][i])
                    G_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_055_065_index][i])
                    G_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_055_065_index][i])
                    G_Mag_Array.append(j['details']['mag'])
                    G_Mag_Err_Array.append(j['details']['mag_err'])
                    G_Aperture_Array.append(j['aperture'])


                elif j['details']["filter_name"] == "r":
                    #print("r")
                    R_Data_Array.append(j['cutout'].data)
                    R_RA_Array.append(j['details']['ra'])
                    R_DEC_Array.append(j['details']['dec'])
                    R_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_055_065_index][i])
                    R_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_055_065_index][i])
                    R_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_055_065_index][i])
                    R_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_055_065_index][i])
                    R_Mag_Array.append(j['details']['mag'])
                    R_Mag_Err_Array.append(j['details']['mag_err'])
                    R_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "i":
                    I_Data_Array.append(j['cutout'].data)
                    I_RA_Array.append(j['details']['ra'])
                    I_DEC_Array.append(j['details']['dec'])
                    I_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_055_065_index][i])
                    I_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_055_065_index][i])
                    I_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_055_065_index][i])
                    I_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_055_065_index][i])
                    I_Mag_Array.append(j['details']['mag'])
                    I_Mag_Err_Array.append(j['details']['mag_err'])
                    I_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "z":
                    Z_Data_Array.append(j['cutout'].data)
                    Z_RA_Array.append(j['details']['ra'])
                    Z_DEC_Array.append(j['details']['dec'])
                    Z_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_055_065_index][i])
                    Z_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_055_065_index][i])
                    Z_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_055_065_index][i])
                    Z_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_055_065_index][i])
                    Z_Mag_Array.append(j['details']['mag'])
                    Z_Mag_Err_Array.append(j['details']['mag_err'])
                    Z_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "y":
                    Y_Data_Array.append(j['cutout'].data)
                    Y_RA_Array.append(j['details']['ra'])
                    Y_DEC_Array.append(j['details']['dec'])
                    Y_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_055_065_index][i])
                    Y_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_055_065_index][i])
                    Y_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_055_065_index][i])
                    Y_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_055_065_index][i])
                    Y_Mag_Array.append(j['details']['mag'])
                    Y_Mag_Err_Array.append(j['details']['mag_err'])
                    Y_Aperture_Array.append(j['aperture'])
                    

In [ ]:
Table_start_a = ([(G_Data_Array[0])])
Table_start_b = ([(G_RA_Array[0])])
Table_start_c = ([(G_DEC_Array[0])])
Table_start_d = ([(G_Detect_ID[0])])
Table_start_e = ([(G_Redshift_Array[0])])
Table_start_f = ([(G_Mag_Array[0])])
Table_start_g = ([(G_Mag_Err_Array[0])])
Table_start_h = ([(G_Aperture_Array[0])])
Table_start_i = ([(G_Redshift_Err_Array[0])])
Table_start_j = ([(G_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("G_Data","G_RA","G_DEC", "G_Detect_ID", "G_Z", "G_Mag", "G_Mag_Err", "G_Aperture_Array", "G_Z_Err", "G_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(G_RA_Array),1):
    a = ([(G_Data_Array[i])])
    b = ([(G_RA_Array[i])])
    c = ([(G_DEC_Array[i])])
    d = ([(G_Detect_ID[i])])
    e = ([(G_Redshift_Array[i])])
    f = ([(G_Mag_Array[i])])
    g = ([(G_Mag_Err_Array[i])])
    h = ([(G_Aperture_Array[i])])
    ii = ([(G_Redshift_Err_Array[i])])
    j = ([(G_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_055_065_G.fits", overwrite = True)

In [ ]:
Table_start_a = ([(R_Data_Array[0])])
Table_start_b = ([(R_RA_Array[0])])
Table_start_c = ([(R_DEC_Array[0])])
Table_start_d = ([(R_Detect_ID[0])])
Table_start_e = ([(R_Redshift_Array[0])])
Table_start_f = ([(R_Mag_Array[0])])
Table_start_g = ([(R_Mag_Err_Array[0])])
Table_start_h = ([(R_Aperture_Array[0])])
Table_start_i = ([(R_Redshift_Err_Array[0])])
Table_start_j = ([(R_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("R_Data","R_RA","R_DEC", "R_Detect_ID", "R_Z", "R_Mag", "R_Mag_Err", "R_Aperture_Array", "R_Z_Err", "R_MgII_SNR"))


In [ ]:
print(len(G_Data_Array), len(R_Data_Array), len(I_Data_Array), len(Z_Data_Array), len(Y_Data_Array))

In [ ]:
for i in np.arange(1,len(R_RA_Array),1):
    a = ([(R_Data_Array[i])])
    b = ([(R_RA_Array[i])])
    c = ([(R_DEC_Array[i])])
    d = ([(R_Detect_ID[i])])
    e = ([(R_Redshift_Array[i])])
    f = ([(R_Mag_Array[i])])
    g = ([(R_Mag_Err_Array[i])])
    h = ([(R_Aperture_Array[i])])
    ii = ([(R_Redshift_Err_Array[i])])
    j = ([(R_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_055_065_R.fits", overwrite = True)

In [ ]:
Table_start_a = ([(I_Data_Array[0])])
Table_start_b = ([(I_RA_Array[0])])
Table_start_c = ([(I_DEC_Array[0])])
Table_start_d = ([(I_Detect_ID[0])])
Table_start_e = ([(I_Redshift_Array[0])])
Table_start_f = ([(I_Mag_Array[0])])
Table_start_g = ([(I_Mag_Err_Array[0])])
Table_start_h = ([(I_Aperture_Array[0])])
Table_start_i = ([(I_Redshift_Err_Array[0])])
Table_start_j = ([(I_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("I_Data","I_RA","I_DEC", "I_Detect_ID", "I_Z", "I_Mag", "I_Mag_Err", "I_Aperture_Array", "I_Z_Err", "I_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(I_RA_Array),1):
    a = ([(I_Data_Array[i])])
    b = ([(I_RA_Array[i])])
    c = ([(I_DEC_Array[i])])
    d = ([(I_Detect_ID[i])])
    e = ([(I_Redshift_Array[i])])
    f = ([(I_Mag_Array[i])])
    g = ([(I_Mag_Err_Array[i])])
    h = ([(I_Aperture_Array[i])])
    ii = ([(I_Redshift_Err_Array[i])])
    j = ([(I_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_055_065_I.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Z_Data_Array[0])])
Table_start_b = ([(Z_RA_Array[0])])
Table_start_c = ([(Z_DEC_Array[0])])
Table_start_d = ([(Z_Detect_ID[0])])
Table_start_e = ([(Z_Redshift_Array[0])])
Table_start_f = ([(Z_Mag_Array[0])])
Table_start_g = ([(Z_Mag_Err_Array[0])])
Table_start_h = ([(Z_Aperture_Array[0])])
Table_start_i = ([(Z_Redshift_Err_Array[0])])
Table_start_j = ([(Z_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Z_Data","Z_RA","Z_DEC", "Z_Detect_ID", "Z_Z", "Z_Mag", "Z_Mag_Err", "Z_Aperture_Array", "Z_Z_Err", "Z_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Z_RA_Array),1):
    a = ([(Z_Data_Array[i])])
    b = ([(Z_RA_Array[i])])
    c = ([(Z_DEC_Array[i])])
    d = ([(Z_Detect_ID[i])])
    e = ([(Z_Redshift_Array[i])])
    f = ([(Z_Mag_Array[i])])
    g = ([(Z_Mag_Err_Array[i])])
    h = ([(Z_Aperture_Array[i])])
    ii = ([(Z_Redshift_Err_Array[i])])
    j = ([(Z_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_055_065_Z.fits", overwrite = True)


In [ ]:
Table_start_a = ([(Y_Data_Array[0])])
Table_start_b = ([(Y_RA_Array[0])])
Table_start_c = ([(Y_DEC_Array[0])])
Table_start_d = ([(Y_Detect_ID[0])])
Table_start_e = ([(Y_Redshift_Array[0])])
Table_start_f = ([(Y_Mag_Array[0])])
Table_start_g = ([(Y_Mag_Err_Array[0])])
Table_start_h = ([(Y_Aperture_Array[0])])
Table_start_i = ([(Y_Redshift_Err_Array[0])])
Table_start_j = ([(Y_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Y_Data","Y_RA","Y_DEC", "Y_Detect_ID", "Y_Z", "Y_Mag", "Y_Mag_Err", "Y_Aperture_Array", "Y_Z_Err", "Y_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_055_065_Y.fits", overwrite = True)

<hr style="border:4px solid blue">

In [ ]:
G_Data_Array = []
G_RA_Array = []
G_DEC_Array = []
G_Detect_ID = []
G_Redshift_Array = []
G_Redshift_Err_Array = []
G_SNR_Array = []
G_Mag_Array = []
G_Mag_Err_Array = []
G_Aperture_Array = []

R_Data_Array = []
R_RA_Array = []
R_DEC_Array = []
R_Detect_ID = []
R_Redshift_Array = []
R_Redshift_Err_Array = []
R_SNR_Array = []
R_Mag_Array = []
R_Mag_Err_Array = []
R_Aperture_Array = []

I_Data_Array = []
I_RA_Array = []
I_DEC_Array = []
I_Detect_ID = []
I_Redshift_Array = []
I_Redshift_Err_Array = []
I_SNR_Array = []
I_Mag_Array = []
I_Mag_Err_Array = []
I_Aperture_Array = []

Z_Data_Array = []
Z_RA_Array = []
Z_DEC_Array = []
Z_Detect_ID = []
Z_Redshift_Array = []
Z_Redshift_Err_Array = []
Z_SNR_Array = []
Z_Mag_Array = []
Z_Mag_Err_Array = []
Z_Aperture_Array = []

Y_Data_Array = []
Y_RA_Array = []
Y_DEC_Array = []
Y_Detect_ID = []
Y_Redshift_Array = []
Y_Redshift_Err_Array = []
Y_SNR_Array = []
Y_Mag_Array = []
Y_Mag_Err_Array = []
Y_Aperture_Array = []



for i in np.arange(len(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_065_075_index])): 

#for i in np.arange(10): 
    coord7 = SkyCoord(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_065_075_index][i], Data_Confirmed_Redshift_And_AGN_Status["DEC"][z_065_075_index][i], unit = 'deg')
    cutouts7 = catlib.get_cutouts(position=coord7,radius=7.,aperture=1.5,dynamic=False,first=False,nudge=False,filter=None)

    #print(cutouts7)
    if len(cutouts7) > 0:
        cutout = cutouts7[0] 
        #print(cutout['instrument'])
        
        if cutout['instrument'] == "HSC SSP":
            for j in cutouts7:
                #print(j['details']["filter_name"])
                
                if j['details']["filter_name"] == "g":
                    #print("g")
                    #print(j['cutout'].error)
                    G_Data_Array.append(j['cutout'].data)
                    G_RA_Array.append(j['details']['ra'])
                    G_DEC_Array.append(j['details']['dec'])
                    G_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_065_075_index][i])
                    G_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_065_075_index][i])
                    G_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_065_075_index][i])
                    G_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_065_075_index][i])
                    G_Mag_Array.append(j['details']['mag'])
                    G_Mag_Err_Array.append(j['details']['mag_err'])
                    G_Aperture_Array.append(j['aperture'])


                elif j['details']["filter_name"] == "r":
                    #print("r")
                    R_Data_Array.append(j['cutout'].data)
                    R_RA_Array.append(j['details']['ra'])
                    R_DEC_Array.append(j['details']['dec'])
                    R_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_065_075_index][i])
                    R_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_065_075_index][i])
                    R_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_065_075_index][i])
                    R_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_065_075_index][i])
                    R_Mag_Array.append(j['details']['mag'])
                    R_Mag_Err_Array.append(j['details']['mag_err'])
                    R_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "i":
                    I_Data_Array.append(j['cutout'].data)
                    I_RA_Array.append(j['details']['ra'])
                    I_DEC_Array.append(j['details']['dec'])
                    I_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_065_075_index][i])
                    I_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_065_075_index][i])
                    I_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_065_075_index][i])
                    I_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_065_075_index][i])
                    I_Mag_Array.append(j['details']['mag'])
                    I_Mag_Err_Array.append(j['details']['mag_err'])
                    I_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "z":
                    Z_Data_Array.append(j['cutout'].data)
                    Z_RA_Array.append(j['details']['ra'])
                    Z_DEC_Array.append(j['details']['dec'])
                    Z_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_065_075_index][i])
                    Z_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_065_075_index][i])
                    Z_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_065_075_index][i])
                    Z_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_065_075_index][i])
                    Z_Mag_Array.append(j['details']['mag'])
                    Z_Mag_Err_Array.append(j['details']['mag_err'])
                    Z_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "y":
                    Y_Data_Array.append(j['cutout'].data)
                    Y_RA_Array.append(j['details']['ra'])
                    Y_DEC_Array.append(j['details']['dec'])
                    Y_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_065_075_index][i])
                    Y_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_065_075_index][i])
                    Y_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_065_075_index][i])
                    Y_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_065_075_index][i])
                    Y_Mag_Array.append(j['details']['mag'])
                    Y_Mag_Err_Array.append(j['details']['mag_err'])
                    Y_Aperture_Array.append(j['aperture'])

                    

In [ ]:
Table_start_a = ([(G_Data_Array[0])])
Table_start_b = ([(G_RA_Array[0])])
Table_start_c = ([(G_DEC_Array[0])])
Table_start_d = ([(G_Detect_ID[0])])
Table_start_e = ([(G_Redshift_Array[0])])
Table_start_f = ([(G_Mag_Array[0])])
Table_start_g = ([(G_Mag_Err_Array[0])])
Table_start_h = ([(G_Aperture_Array[0])])
Table_start_i = ([(G_Redshift_Err_Array[0])])
Table_start_j = ([(G_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("G_Data","G_RA","G_DEC", "G_Detect_ID", "G_Z", "G_Mag", "G_Mag_Err", "G_Aperture_Array", "G_Z_Err", "G_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(G_RA_Array),1):
    a = ([(G_Data_Array[i])])
    b = ([(G_RA_Array[i])])
    c = ([(G_DEC_Array[i])])
    d = ([(G_Detect_ID[i])])
    e = ([(G_Redshift_Array[i])])
    f = ([(G_Mag_Array[i])])
    g = ([(G_Mag_Err_Array[i])])
    h = ([(G_Aperture_Array[i])])
    ii = ([(G_Redshift_Err_Array[i])])
    j = ([(G_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_065_075_G.fits", overwrite = True)

In [ ]:
Table_start_a = ([(R_Data_Array[0])])
Table_start_b = ([(R_RA_Array[0])])
Table_start_c = ([(R_DEC_Array[0])])
Table_start_d = ([(R_Detect_ID[0])])
Table_start_e = ([(R_Redshift_Array[0])])
Table_start_f = ([(R_Mag_Array[0])])
Table_start_g = ([(R_Mag_Err_Array[0])])
Table_start_h = ([(R_Aperture_Array[0])])
Table_start_i = ([(R_Redshift_Err_Array[0])])
Table_start_j = ([(R_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("R_Data","R_RA","R_DEC", "R_Detect_ID", "R_Z", "R_Mag", "R_Mag_Err", "R_Aperture_Array", "R_Z_Err", "R_MgII_SNR"))


In [ ]:
print(len(G_Data_Array), len(R_Data_Array), len(I_Data_Array), len(Z_Data_Array), len(Y_Data_Array))

In [ ]:
for i in np.arange(1,len(R_RA_Array),1):
    a = ([(R_Data_Array[i])])
    b = ([(R_RA_Array[i])])
    c = ([(R_DEC_Array[i])])
    d = ([(R_Detect_ID[i])])
    e = ([(R_Redshift_Array[i])])
    f = ([(R_Mag_Array[i])])
    g = ([(R_Mag_Err_Array[i])])
    h = ([(R_Aperture_Array[i])])
    ii = ([(R_Redshift_Err_Array[i])])
    j = ([(R_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_065_075_R.fits", overwrite = True)

In [ ]:
Table_start_a = ([(I_Data_Array[0])])
Table_start_b = ([(I_RA_Array[0])])
Table_start_c = ([(I_DEC_Array[0])])
Table_start_d = ([(I_Detect_ID[0])])
Table_start_e = ([(I_Redshift_Array[0])])
Table_start_f = ([(I_Mag_Array[0])])
Table_start_g = ([(I_Mag_Err_Array[0])])
Table_start_h = ([(I_Aperture_Array[0])])
Table_start_i = ([(I_Redshift_Err_Array[0])])
Table_start_j = ([(I_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("I_Data","I_RA","I_DEC", "I_Detect_ID", "I_Z", "I_Mag", "I_Mag_Err", "I_Aperture_Array", "I_Z_Err", "I_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(I_RA_Array),1):
    a = ([(I_Data_Array[i])])
    b = ([(I_RA_Array[i])])
    c = ([(I_DEC_Array[i])])
    d = ([(I_Detect_ID[i])])
    e = ([(I_Redshift_Array[i])])
    f = ([(I_Mag_Array[i])])
    g = ([(I_Mag_Err_Array[i])])
    h = ([(I_Aperture_Array[i])])
    ii = ([(I_Redshift_Err_Array[i])])
    j = ([(I_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_065_075_I.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Z_Data_Array[0])])
Table_start_b = ([(Z_RA_Array[0])])
Table_start_c = ([(Z_DEC_Array[0])])
Table_start_d = ([(Z_Detect_ID[0])])
Table_start_e = ([(Z_Redshift_Array[0])])
Table_start_f = ([(Z_Mag_Array[0])])
Table_start_g = ([(Z_Mag_Err_Array[0])])
Table_start_h = ([(Z_Aperture_Array[0])])
Table_start_i = ([(Z_Redshift_Err_Array[0])])
Table_start_j = ([(Z_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Z_Data","Z_RA","Z_DEC", "Z_Detect_ID", "Z_Z", "Z_Mag", "Z_Mag_Err", "Z_Aperture_Array", "Z_Z_Err", "Z_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Z_RA_Array),1):
    a = ([(Z_Data_Array[i])])
    b = ([(Z_RA_Array[i])])
    c = ([(Z_DEC_Array[i])])
    d = ([(Z_Detect_ID[i])])
    e = ([(Z_Redshift_Array[i])])
    f = ([(Z_Mag_Array[i])])
    g = ([(Z_Mag_Err_Array[i])])
    h = ([(Z_Aperture_Array[i])])
    ii = ([(Z_Redshift_Err_Array[i])])
    j = ([(Z_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_065_075_Z.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Y_Data_Array[0])])
Table_start_b = ([(Y_RA_Array[0])])
Table_start_c = ([(Y_DEC_Array[0])])
Table_start_d = ([(Y_Detect_ID[0])])
Table_start_e = ([(Y_Redshift_Array[0])])
Table_start_f = ([(Y_Mag_Array[0])])
Table_start_g = ([(Y_Mag_Err_Array[0])])
Table_start_h = ([(Y_Aperture_Array[0])])
Table_start_i = ([(Y_Redshift_Err_Array[0])])
Table_start_j = ([(Y_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Y_Data","Y_RA","Y_DEC", "Y_Detect_ID", "Y_Z", "Y_Mag", "Y_Mag_Err", "Y_Aperture_Array", "Y_Z_Err", "Y_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_065_075_Y.fits", overwrite = True)

<hr style="border:4px solid blue">

In [ ]:
G_Data_Array = []
G_RA_Array = []
G_DEC_Array = []
G_Detect_ID = []
G_Redshift_Array = []
G_Redshift_Err_Array = []
G_SNR_Array = []
G_Mag_Array = []
G_Mag_Err_Array = []
G_Aperture_Array = []

R_Data_Array = []
R_RA_Array = []
R_DEC_Array = []
R_Detect_ID = []
R_Redshift_Array = []
R_Redshift_Err_Array = []
R_SNR_Array = []
R_Mag_Array = []
R_Mag_Err_Array = []
R_Aperture_Array = []

I_Data_Array = []
I_RA_Array = []
I_DEC_Array = []
I_Detect_ID = []
I_Redshift_Array = []
I_Redshift_Err_Array = []
I_SNR_Array = []
I_Mag_Array = []
I_Mag_Err_Array = []
I_Aperture_Array = []

Z_Data_Array = []
Z_RA_Array = []
Z_DEC_Array = []
Z_Detect_ID = []
Z_Redshift_Array = []
Z_Redshift_Err_Array = []
Z_SNR_Array = []
Z_Mag_Array = []
Z_Mag_Err_Array = []
Z_Aperture_Array = []

Y_Data_Array = []
Y_RA_Array = []
Y_DEC_Array = []
Y_Detect_ID = []
Y_Redshift_Array = []
Y_Redshift_Err_Array = []
Y_SNR_Array = []
Y_Mag_Array = []
Y_Mag_Err_Array = []
Y_Aperture_Array = []




for i in np.arange(len(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_075_085_index])): 

#for i in np.arange(10): 
    coord7 = SkyCoord(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_075_085_index][i], Data_Confirmed_Redshift_And_AGN_Status["DEC"][z_075_085_index][i], unit = 'deg')
    cutouts7 = catlib.get_cutouts(position=coord7,radius=7.,aperture=1.5,dynamic=False,first=False,nudge=False,filter=None)

    #print(cutouts7)
    if len(cutouts7) > 0:
        cutout = cutouts7[0] 
        #print(cutout['instrument'])
        
        if cutout['instrument'] == "HSC SSP":
            for j in cutouts7:
                #print(j['details']["filter_name"])
                
                if j['details']["filter_name"] == "g":
                    #print("g")
                    #print(j['cutout'].error)
                    G_Data_Array.append(j['cutout'].data)
                    G_RA_Array.append(j['details']['ra'])
                    G_DEC_Array.append(j['details']['dec'])
                    G_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_075_085_index][i])
                    G_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_075_085_index][i])
                    G_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_075_085_index][i])
                    G_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_075_085_index][i])
                    G_Mag_Array.append(j['details']['mag'])
                    G_Mag_Err_Array.append(j['details']['mag_err'])
                    G_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "r":
                    #print("r")
                    R_Data_Array.append(j['cutout'].data)
                    R_RA_Array.append(j['details']['ra'])
                    R_DEC_Array.append(j['details']['dec'])
                    R_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_075_085_index][i])
                    R_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_075_085_index][i])
                    R_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_075_085_index][i])
                    R_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_075_085_index][i])
                    R_Mag_Array.append(j['details']['mag'])
                    R_Mag_Err_Array.append(j['details']['mag_err'])
                    R_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "i":
                    I_Data_Array.append(j['cutout'].data)
                    I_RA_Array.append(j['details']['ra'])
                    I_DEC_Array.append(j['details']['dec'])
                    I_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_075_085_index][i])
                    I_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_075_085_index][i])
                    I_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_075_085_index][i])
                    I_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_075_085_index][i])
                    I_Mag_Array.append(j['details']['mag'])
                    I_Mag_Err_Array.append(j['details']['mag_err'])
                    I_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "z":
                    Z_Data_Array.append(j['cutout'].data)
                    Z_RA_Array.append(j['details']['ra'])
                    Z_DEC_Array.append(j['details']['dec'])
                    Z_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_075_085_index][i])
                    Z_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_075_085_index][i])
                    Z_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_075_085_index][i])
                    Z_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_075_085_index][i])
                    Z_Mag_Array.append(j['details']['mag'])
                    Z_Mag_Err_Array.append(j['details']['mag_err'])
                    Z_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "y":
                    Y_Data_Array.append(j['cutout'].data)
                    Y_RA_Array.append(j['details']['ra'])
                    Y_DEC_Array.append(j['details']['dec'])
                    Y_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_075_085_index][i])
                    Y_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_075_085_index][i])
                    Y_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_075_085_index][i])
                    Y_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_075_085_index][i])
                    Y_Mag_Array.append(j['details']['mag'])
                    Y_Mag_Err_Array.append(j['details']['mag_err'])
                    Y_Aperture_Array.append(j['aperture'])



In [ ]:
Table_start_a = ([(G_Data_Array[0])])
Table_start_b = ([(G_RA_Array[0])])
Table_start_c = ([(G_DEC_Array[0])])
Table_start_d = ([(G_Detect_ID[0])])
Table_start_e = ([(G_Redshift_Array[0])])
Table_start_f = ([(G_Mag_Array[0])])
Table_start_g = ([(G_Mag_Err_Array[0])])
Table_start_h = ([(G_Aperture_Array[0])])
Table_start_i = ([(G_Redshift_Err_Array[0])])
Table_start_j = ([(G_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("G_Data","G_RA","G_DEC", "G_Detect_ID", "G_Z", "G_Mag", "G_Mag_Err", "G_Aperture_Array", "G_Z_Err", "G_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(G_RA_Array),1):
    a = ([(G_Data_Array[i])])
    b = ([(G_RA_Array[i])])
    c = ([(G_DEC_Array[i])])
    d = ([(G_Detect_ID[i])])
    e = ([(G_Redshift_Array[i])])
    f = ([(G_Mag_Array[i])])
    g = ([(G_Mag_Err_Array[i])])
    h = ([(G_Aperture_Array[i])])
    ii = ([(G_Redshift_Err_Array[i])])
    j = ([(G_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_075_085_G.fits", overwrite = True)

In [ ]:
Table_start_a = ([(R_Data_Array[0])])
Table_start_b = ([(R_RA_Array[0])])
Table_start_c = ([(R_DEC_Array[0])])
Table_start_d = ([(R_Detect_ID[0])])
Table_start_e = ([(R_Redshift_Array[0])])
Table_start_f = ([(R_Mag_Array[0])])
Table_start_g = ([(R_Mag_Err_Array[0])])
Table_start_h = ([(R_Aperture_Array[0])])
Table_start_i = ([(R_Redshift_Err_Array[0])])
Table_start_j = ([(R_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("R_Data","R_RA","R_DEC", "R_Detect_ID", "R_Z", "R_Mag", "R_Mag_Err", "R_Aperture_Array", "R_Z_Err", "R_MgII_SNR"))


In [ ]:
print(len(G_Data_Array), len(R_Data_Array), len(I_Data_Array), len(Z_Data_Array), len(Y_Data_Array))

In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_075_085_R.fits", overwrite = True)

In [ ]:
Table_start_a = ([(I_Data_Array[0])])
Table_start_b = ([(I_RA_Array[0])])
Table_start_c = ([(I_DEC_Array[0])])
Table_start_d = ([(I_Detect_ID[0])])
Table_start_e = ([(I_Redshift_Array[0])])
Table_start_f = ([(I_Mag_Array[0])])
Table_start_g = ([(I_Mag_Err_Array[0])])
Table_start_h = ([(I_Aperture_Array[0])])
Table_start_i = ([(I_Redshift_Err_Array[0])])
Table_start_j = ([(I_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("I_Data","I_RA","I_DEC", "I_Detect_ID", "I_Z", "I_Mag", "I_Mag_Err", "I_Aperture_Array", "I_Z_Err", "I_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(I_RA_Array),1):
    a = ([(I_Data_Array[i])])
    b = ([(I_RA_Array[i])])
    c = ([(I_DEC_Array[i])])
    d = ([(I_Detect_ID[i])])
    e = ([(I_Redshift_Array[i])])
    f = ([(I_Mag_Array[i])])
    g = ([(I_Mag_Err_Array[i])])
    h = ([(I_Aperture_Array[i])])
    ii = ([(I_Redshift_Err_Array[i])])
    j = ([(I_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_075_085_I.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Z_Data_Array[0])])
Table_start_b = ([(Z_RA_Array[0])])
Table_start_c = ([(Z_DEC_Array[0])])
Table_start_d = ([(Z_Detect_ID[0])])
Table_start_e = ([(Z_Redshift_Array[0])])
Table_start_f = ([(Z_Mag_Array[0])])
Table_start_g = ([(Z_Mag_Err_Array[0])])
Table_start_h = ([(Z_Aperture_Array[0])])
Table_start_i = ([(Z_Redshift_Err_Array[0])])
Table_start_j = ([(Z_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Z_Data","Z_RA","Z_DEC", "Z_Detect_ID", "Z_Z", "Z_Mag", "Z_Mag_Err", "Z_Aperture_Array", "Z_Z_Err", "Z_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Z_RA_Array),1):
    a = ([(Z_Data_Array[i])])
    b = ([(Z_RA_Array[i])])
    c = ([(Z_DEC_Array[i])])
    d = ([(Z_Detect_ID[i])])
    e = ([(Z_Redshift_Array[i])])
    f = ([(Z_Mag_Array[i])])
    g = ([(Z_Mag_Err_Array[i])])
    h = ([(Z_Aperture_Array[i])])
    ii = ([(Z_Redshift_Err_Array[i])])
    j = ([(Z_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_075_085_Z.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Y_Data_Array[0])])
Table_start_b = ([(Y_RA_Array[0])])
Table_start_c = ([(Y_DEC_Array[0])])
Table_start_d = ([(Y_Detect_ID[0])])
Table_start_e = ([(Y_Redshift_Array[0])])
Table_start_f = ([(Y_Mag_Array[0])])
Table_start_g = ([(Y_Mag_Err_Array[0])])
Table_start_h = ([(Y_Aperture_Array[0])])
Table_start_i = ([(Y_Redshift_Err_Array[0])])
Table_start_j = ([(Y_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Y_Data","Y_RA","Y_DEC", "Y_Detect_ID", "Y_Z", "Y_Mag", "Y_Mag_Err", "Y_Aperture_Array", "Y_Z_Err", "Y_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_075_085_Y.fits", overwrite = True)

<hr style="border:4px solid blue">

In [ ]:
G_Data_Array = []
G_RA_Array = []
G_DEC_Array = []
G_Detect_ID = []
G_Redshift_Array = []
G_Redshift_Err_Array = []
G_SNR_Array = []
G_Mag_Array = []
G_Mag_Err_Array = []
G_Aperture_Array = []

R_Data_Array = []
R_RA_Array = []
R_DEC_Array = []
R_Detect_ID = []
R_Redshift_Array = []
R_Redshift_Err_Array = []
R_SNR_Array = []
R_Mag_Array = []
R_Mag_Err_Array = []
R_Aperture_Array = []

I_Data_Array = []
I_RA_Array = []
I_DEC_Array = []
I_Detect_ID = []
I_Redshift_Array = []
I_Redshift_Err_Array = []
I_SNR_Array = []
I_Mag_Array = []
I_Mag_Err_Array = []
I_Aperture_Array = []

Z_Data_Array = []
Z_RA_Array = []
Z_DEC_Array = []
Z_Detect_ID = []
Z_Redshift_Array = []
Z_Redshift_Err_Array = []
Z_SNR_Array = []
Z_Mag_Array = []
Z_Mag_Err_Array = []
Z_Aperture_Array = []

Y_Data_Array = []
Y_RA_Array = []
Y_DEC_Array = []
Y_Detect_ID = []
Y_Redshift_Array = []
Y_Redshift_Err_Array = []
Y_SNR_Array = []
Y_Mag_Array = []
Y_Mag_Err_Array = []
Y_Aperture_Array = []


for i in np.arange(len(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_085_096_index])): 

#for i in np.arange(10): 
    coord7 = SkyCoord(Data_Confirmed_Redshift_And_AGN_Status["RA"][z_085_096_index][i], Data_Confirmed_Redshift_And_AGN_Status["DEC"][z_085_096_index][i], unit = 'deg')
    cutouts7 = catlib.get_cutouts(position=coord7,radius=7.,aperture=1.5,dynamic=False,first=False,nudge=False,filter=None)

    #print(cutouts7)
    if len(cutouts7) > 0:
        cutout = cutouts7[0] 
        #print(cutout['instrument'])
        
        if cutout['instrument'] == "HSC SSP":
            for j in cutouts7:
                #print(j['details']["filter_name"])
                
                if j['details']["filter_name"] == "g":
                    #print("g")
                    #print(j['cutout'].error)
                    G_Data_Array.append(j['cutout'].data)
                    G_RA_Array.append(j['details']['ra'])
                    G_DEC_Array.append(j['details']['dec'])
                    G_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_085_096_index][i])
                    G_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_085_096_index][i])
                    G_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_085_096_index][i])
                    G_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_085_096_index][i])
                    G_Mag_Array.append(j['details']['mag'])
                    G_Mag_Err_Array.append(j['details']['mag_err'])
                    G_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "r":
                    #print("r")
                    R_Data_Array.append(j['cutout'].data)
                    R_RA_Array.append(j['details']['ra'])
                    R_DEC_Array.append(j['details']['dec'])
                    R_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_085_096_index][i])
                    R_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_085_096_index][i])
                    R_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_085_096_index][i])
                    R_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_085_096_index][i])
                    R_Mag_Array.append(j['details']['mag'])
                    R_Mag_Err_Array.append(j['details']['mag_err'])
                    R_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "i":
                    I_Data_Array.append(j['cutout'].data)
                    I_RA_Array.append(j['details']['ra'])
                    I_DEC_Array.append(j['details']['dec'])
                    I_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_085_096_index][i])
                    I_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_085_096_index][i])
                    I_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_085_096_index][i])
                    I_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_085_096_index][i])
                    I_Mag_Array.append(j['details']['mag'])
                    I_Mag_Err_Array.append(j['details']['mag_err'])
                    I_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "z":
                    Z_Data_Array.append(j['cutout'].data)
                    Z_RA_Array.append(j['details']['ra'])
                    Z_DEC_Array.append(j['details']['dec'])
                    Z_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_085_096_index][i])
                    Z_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_085_096_index][i])
                    Z_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_085_096_index][i])
                    Z_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_085_096_index][i])
                    Z_Mag_Array.append(j['details']['mag'])
                    Z_Mag_Err_Array.append(j['details']['mag_err'])
                    Z_Aperture_Array.append(j['aperture'])

                elif j['details']["filter_name"] == "y":
                    Y_Data_Array.append(j['cutout'].data)
                    Y_RA_Array.append(j['details']['ra'])
                    Y_DEC_Array.append(j['details']['dec'])
                    Y_Detect_ID.append(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][z_085_096_index][i])
                    Y_Redshift_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z"][z_085_096_index][i])
                    Y_Redshift_Err_Array.append(Data_Confirmed_Redshift_And_AGN_Status["z_err"][z_085_096_index][i])
                    Y_SNR_Array.append(Data_Confirmed_Redshift_And_AGN_Status["snr_MgII"][z_085_096_index][i])
                    Y_Mag_Array.append(j['details']['mag'])
                    Y_Mag_Err_Array.append(j['details']['mag_err'])
                    Y_Aperture_Array.append(j['aperture'])



In [ ]:
Table_start_a = ([(G_Data_Array[0])])
Table_start_b = ([(G_RA_Array[0])])
Table_start_c = ([(G_DEC_Array[0])])
Table_start_d = ([(G_Detect_ID[0])])
Table_start_e = ([(G_Redshift_Array[0])])
Table_start_f = ([(G_Mag_Array[0])])
Table_start_g = ([(G_Mag_Err_Array[0])])
Table_start_h = ([(G_Aperture_Array[0])])
Table_start_i = ([(G_Redshift_Err_Array[0])])
Table_start_j = ([(G_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("G_Data","G_RA","G_DEC", "G_Detect_ID", "G_Z", "G_Mag", "G_Mag_Err", "G_Aperture_Array", "G_Z_Err", "G_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(G_RA_Array),1):
    a = ([(G_Data_Array[i])])
    b = ([(G_RA_Array[i])])
    c = ([(G_DEC_Array[i])])
    d = ([(G_Detect_ID[i])])
    e = ([(G_Redshift_Array[i])])
    f = ([(G_Mag_Array[i])])
    g = ([(G_Mag_Err_Array[i])])
    h = ([(G_Aperture_Array[i])])
    ii = ([(G_Redshift_Err_Array[i])])
    j = ([(G_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_085_096_G.fits", overwrite = True)

In [ ]:
Table_start_a = ([(R_Data_Array[0])])
Table_start_b = ([(R_RA_Array[0])])
Table_start_c = ([(R_DEC_Array[0])])
Table_start_d = ([(R_Detect_ID[0])])
Table_start_e = ([(R_Redshift_Array[0])])
Table_start_f = ([(R_Mag_Array[0])])
Table_start_g = ([(R_Mag_Err_Array[0])])
Table_start_h = ([(R_Aperture_Array[0])])
Table_start_i = ([(R_Redshift_Err_Array[0])])
Table_start_j = ([(R_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("R_Data","R_RA","R_DEC", "R_Detect_ID", "R_Z", "R_Mag", "R_Mag_Err", "R_Aperture_Array", "R_Z_Err", "R_MgII_SNR"))


In [ ]:
print(len(G_Data_Array), len(R_Data_Array), len(I_Data_Array), len(Z_Data_Array), len(Y_Data_Array))

In [ ]:
for i in np.arange(1,len(R_RA_Array),1):
    a = ([(R_Data_Array[i])])
    b = ([(R_RA_Array[i])])
    c = ([(R_DEC_Array[i])])
    d = ([(R_Detect_ID[i])])
    e = ([(R_Redshift_Array[i])])
    f = ([(R_Mag_Array[i])])
    g = ([(R_Mag_Err_Array[i])])
    h = ([(R_Aperture_Array[i])])
    ii = ([(R_Redshift_Err_Array[i])])
    j = ([(R_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_085_096_R.fits", overwrite = True)

In [ ]:
Table_start_a = ([(I_Data_Array[0])])
Table_start_b = ([(I_RA_Array[0])])
Table_start_c = ([(I_DEC_Array[0])])
Table_start_d = ([(I_Detect_ID[0])])
Table_start_e = ([(I_Redshift_Array[0])])
Table_start_f = ([(I_Mag_Array[0])])
Table_start_g = ([(I_Mag_Err_Array[0])])
Table_start_h = ([(I_Aperture_Array[0])])
Table_start_i = ([(I_Redshift_Err_Array[0])])
Table_start_j = ([(I_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("I_Data","I_RA","I_DEC", "I_Detect_ID", "I_Z", "I_Mag", "I_Mag_Err", "I_Aperture_Array", "I_Z_Err", "I_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(I_RA_Array),1):
    a = ([(I_Data_Array[i])])
    b = ([(I_RA_Array[i])])
    c = ([(I_DEC_Array[i])])
    d = ([(I_Detect_ID[i])])
    e = ([(I_Redshift_Array[i])])
    f = ([(I_Mag_Array[i])])
    g = ([(I_Mag_Err_Array[i])])
    h = ([(I_Aperture_Array[i])])
    ii = ([(I_Redshift_Err_Array[i])])
    j = ([(I_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_085_096_I.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Z_Data_Array[0])])
Table_start_b = ([(Z_RA_Array[0])])
Table_start_c = ([(Z_DEC_Array[0])])
Table_start_d = ([(Z_Detect_ID[0])])
Table_start_e = ([(Z_Redshift_Array[0])])
Table_start_f = ([(Z_Mag_Array[0])])
Table_start_g = ([(Z_Mag_Err_Array[0])])
Table_start_h = ([(Z_Aperture_Array[0])])
Table_start_i = ([(Z_Redshift_Err_Array[0])])
Table_start_j = ([(Z_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Z_Data","Z_RA","Z_DEC", "Z_Detect_ID", "Z_Z", "Z_Mag", "Z_Mag_Err", "Z_Aperture_Array", "Z_Z_Err", "Z_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Z_RA_Array),1):
    a = ([(Z_Data_Array[i])])
    b = ([(Z_RA_Array[i])])
    c = ([(Z_DEC_Array[i])])
    d = ([(Z_Detect_ID[i])])
    e = ([(Z_Redshift_Array[i])])
    f = ([(Z_Mag_Array[i])])
    g = ([(Z_Mag_Err_Array[i])])
    h = ([(Z_Aperture_Array[i])])
    ii = ([(Z_Redshift_Err_Array[i])])
    j = ([(Z_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_085_096_Z.fits", overwrite = True)

In [ ]:
Table_start_a = ([(Y_Data_Array[0])])
Table_start_b = ([(Y_RA_Array[0])])
Table_start_c = ([(Y_DEC_Array[0])])
Table_start_d = ([(Y_Detect_ID[0])])
Table_start_e = ([(Y_Redshift_Array[0])])
Table_start_f = ([(Y_Mag_Array[0])])
Table_start_g = ([(Y_Mag_Err_Array[0])])
Table_start_h = ([(Y_Aperture_Array[0])])
Table_start_i = ([(Y_Redshift_Err_Array[0])])
Table_start_j = ([(Y_SNR_Array[0])])


Table_b = QTable([Table_start_a, Table_start_b, Table_start_c, Table_start_d, Table_start_e, Table_start_f, Table_start_g, Table_start_h, Table_start_i, Table_start_j], names=("Y_Data","Y_RA","Y_DEC", "Y_Detect_ID", "Y_Z", "Y_Mag", "Y_Mag_Err", "Y_Aperture_Array", "Y_Z_Err", "Y_MgII_SNR"))


In [ ]:
for i in np.arange(1,len(Y_RA_Array),1):
    a = ([(Y_Data_Array[i])])
    b = ([(Y_RA_Array[i])])
    c = ([(Y_DEC_Array[i])])
    d = ([(Y_Detect_ID[i])])
    e = ([(Y_Redshift_Array[i])])
    f = ([(Y_Mag_Array[i])])
    g = ([(Y_Mag_Err_Array[i])])
    h = ([(Y_Aperture_Array[i])])
    ii = ([(Y_Redshift_Err_Array[i])])
    j = ([(Y_SNR_Array[i])])

    Table_b.add_row([a,b,c,d,e,f,g,h,ii,j])

In [ ]:
Table_b.write("/home/jovyan/work/stampede3/AGN-Black-Hole-Research-Full-Catalog/Table_Filters_New_Catalog_085_096_Y.fits", overwrite = True)

<hr style="border:4px solid blue">

## <font color='#e55730' size=3 >Opening Files</font>

In [ ]:
G_File_Filters_025_035 = "Table_Filters_New_Catalog_025_035_G.fits"
R_File_Filters_025_035 = "Table_Filters_New_Catalog_025_035_R.fits"
I_File_Filters_025_035 = "Table_Filters_New_Catalog_025_035_I.fits"
Z_File_Filters_025_035 = "Table_Filters_New_Catalog_025_035_Z.fits"
Y_File_Filters_025_035 = "Table_Filters_New_Catalog_025_035_Y.fits"



G_File_Filters_035_045 = "Table_Filters_New_Catalog_035_045_G.fits"
R_File_Filters_035_045 = "Table_Filters_New_Catalog_035_045_R.fits"
I_File_Filters_035_045 = "Table_Filters_New_Catalog_035_045_I.fits"
Z_File_Filters_035_045 = "Table_Filters_New_Catalog_035_045_Z.fits"
Y_File_Filters_035_045 = "Table_Filters_New_Catalog_035_045_Y.fits"


G_File_Filters_045_055 = "Table_Filters_New_Catalog_045_055_G.fits"
R_File_Filters_045_055 = "Table_Filters_New_Catalog_045_055_R.fits"
I_File_Filters_045_055 = "Table_Filters_New_Catalog_045_055_I.fits"
Z_File_Filters_045_055 = "Table_Filters_New_Catalog_045_055_Z.fits"
Y_File_Filters_045_055 = "Table_Filters_New_Catalog_045_055_Y.fits"

G_File_Filters_055_065 = "Table_Filters_New_Catalog_055_065_G.fits"
R_File_Filters_055_065 = "Table_Filters_New_Catalog_055_065_R.fits"
I_File_Filters_055_065 = "Table_Filters_New_Catalog_055_065_I.fits"
Z_File_Filters_055_065 = "Table_Filters_New_Catalog_055_065_Z.fits"
Y_File_Filters_055_065 = "Table_Filters_New_Catalog_055_065_Y.fits"

G_File_Filters_065_075 = "Table_Filters_New_Catalog_065_075_G.fits"
R_File_Filters_065_075 = "Table_Filters_New_Catalog_065_075_R.fits"
I_File_Filters_065_075 = "Table_Filters_New_Catalog_065_075_I.fits"
Z_File_Filters_065_075 = "Table_Filters_New_Catalog_065_075_Z.fits"
Y_File_Filters_065_075 = "Table_Filters_New_Catalog_065_075_Y.fits"

G_File_Filters_075_085 = "Table_Filters_New_Catalog_075_085_G.fits"
R_File_Filters_075_085 = "Table_Filters_New_Catalog_075_085_R.fits"
I_File_Filters_075_085 = "Table_Filters_New_Catalog_075_085_I.fits"
Z_File_Filters_075_085 = "Table_Filters_New_Catalog_075_085_Z.fits"
Y_File_Filters_075_085 = "Table_Filters_New_Catalog_075_085_Y.fits"

G_File_Filters_085_096 = "Table_Filters_New_Catalog_085_096_G.fits"
R_File_Filters_085_096 = "Table_Filters_New_Catalog_085_096_R.fits"
I_File_Filters_085_096 = "Table_Filters_New_Catalog_085_096_I.fits"
Z_File_Filters_085_096 = "Table_Filters_New_Catalog_085_096_Z.fits"
Y_File_Filters_085_096 = "Table_Filters_New_Catalog_085_096_Y.fits"



In [ ]:
G_Filters_025_035 = Table.read(G_File_Filters_025_035, format = 'fits')
R_Filters_025_035 = Table.read(R_File_Filters_025_035, format = 'fits')
I_Filters_025_035 = Table.read(I_File_Filters_025_035, format = 'fits')
Z_Filters_025_035 = Table.read(Z_File_Filters_025_035, format = 'fits')
Y_Filters_025_035 = Table.read(Y_File_Filters_025_035, format = 'fits')


G_Filters_035_045 = Table.read(G_File_Filters_035_045, format = 'fits')
R_Filters_035_045 = Table.read(R_File_Filters_035_045, format = 'fits')
I_Filters_035_045 = Table.read(I_File_Filters_035_045, format = 'fits')
Z_Filters_035_045 = Table.read(Z_File_Filters_035_045, format = 'fits')
Y_Filters_035_045 = Table.read(Y_File_Filters_035_045, format = 'fits')


G_Filters_045_055 = Table.read(G_File_Filters_045_055, format = 'fits')
R_Filters_045_055 = Table.read(R_File_Filters_045_055, format = 'fits')
I_Filters_045_055 = Table.read(I_File_Filters_045_055, format = 'fits')
Z_Filters_045_055 = Table.read(Z_File_Filters_045_055, format = 'fits')
Y_Filters_045_055 = Table.read(Y_File_Filters_045_055, format = 'fits')


G_Filters_055_065 = Table.read(G_File_Filters_055_065, format = 'fits')
R_Filters_055_065 = Table.read(R_File_Filters_055_065, format = 'fits')
I_Filters_055_065 = Table.read(I_File_Filters_055_065, format = 'fits')
Z_Filters_055_065 = Table.read(Z_File_Filters_055_065, format = 'fits')
Y_Filters_055_065 = Table.read(Y_File_Filters_055_065, format = 'fits')


G_Filters_065_075 = Table.read(G_File_Filters_065_075, format = 'fits')
R_Filters_065_075 = Table.read(R_File_Filters_065_075, format = 'fits')
I_Filters_065_075 = Table.read(I_File_Filters_065_075, format = 'fits')
Z_Filters_065_075 = Table.read(Z_File_Filters_065_075, format = 'fits')
Y_Filters_065_075 = Table.read(Y_File_Filters_065_075, format = 'fits')


G_Filters_075_085 = Table.read(G_File_Filters_075_085, format = 'fits')
R_Filters_075_085 = Table.read(R_File_Filters_075_085, format = 'fits')
I_Filters_075_085 = Table.read(I_File_Filters_075_085, format = 'fits')
Z_Filters_075_085 = Table.read(Z_File_Filters_075_085, format = 'fits')
Y_Filters_075_085 = Table.read(Y_File_Filters_075_085, format = 'fits')

G_Filters_085_096 = Table.read(G_File_Filters_085_096, format = 'fits')
R_Filters_085_096 = Table.read(R_File_Filters_085_096, format = 'fits')
I_Filters_085_096 = Table.read(I_File_Filters_085_096, format = 'fits')
Z_Filters_085_096 = Table.read(Z_File_Filters_085_096, format = 'fits')
Y_Filters_085_096 = Table.read(Y_File_Filters_085_096, format = 'fits')

In [ ]:
G_Filters_025_035 = Table(G_Filters_025_035)
R_Filters_025_035 = Table(R_Filters_025_035)
I_Filters_025_035 = Table(I_Filters_025_035)
Z_Filters_025_035 = Table(Z_Filters_025_035)
Y_Filters_025_035 = Table(Y_Filters_025_035)

G_Filters_035_045 = Table(G_Filters_035_045)
R_Filters_035_045 = Table(R_Filters_035_045)
I_Filters_035_045 = Table(I_Filters_035_045)
Z_Filters_035_045 = Table(Z_Filters_035_045)
Y_Filters_035_045 = Table(Y_Filters_035_045)


G_Filters_045_055 = Table(G_Filters_045_055)
R_Filters_045_055 = Table(R_Filters_045_055)
I_Filters_045_055 = Table(I_Filters_045_055)
Z_Filters_045_055 = Table(Z_Filters_045_055)
Y_Filters_045_055 = Table(Y_Filters_045_055)


G_Filters_055_065 = Table(G_Filters_055_065)
R_Filters_055_065 = Table(R_Filters_055_065)
I_Filters_055_065 = Table(I_Filters_055_065)
Z_Filters_055_065 = Table(Z_Filters_055_065)
Y_Filters_055_065 = Table(Y_Filters_055_065)


G_Filters_065_075 = Table(G_Filters_065_075)
R_Filters_065_075 = Table(R_Filters_065_075)
I_Filters_065_075 = Table(I_Filters_065_075)
Z_Filters_065_075 = Table(Z_Filters_065_075)
Y_Filters_065_075 = Table(Y_Filters_065_075)


G_Filters_075_085 = Table(G_Filters_075_085)
R_Filters_075_085 = Table(R_Filters_075_085)
I_Filters_075_085 = Table(I_Filters_075_085)
Z_Filters_075_085 = Table(Z_Filters_075_085)
Y_Filters_075_085 = Table(Y_Filters_075_085)


G_Filters_085_096 = Table(G_Filters_085_096)
R_Filters_085_096 = Table(R_Filters_085_096)
I_Filters_085_096 = Table(I_Filters_085_096)
Z_Filters_085_096 = Table(Z_Filters_085_096)
Y_Filters_085_096 = Table(Y_Filters_085_096)


In [ ]:
G_Data_025_035 = G_Filters_025_035["G_Data"]
G_RA_025_035 = G_Filters_025_035["G_RA"]
G_DEC_025_035 = G_Filters_025_035["G_DEC"]
G_Detect_ID_025_035 = G_Filters_025_035["G_Detect_ID"]
G_Z_025_035 = G_Filters_025_035["G_Z"]

R_Data_025_035 = R_Filters_025_035["G_Data"]
R_RA_025_035 = R_Filters_025_035["G_RA"]
R_DEC_025_035 = R_Filters_025_035["G_DEC"]
R_Detect_ID_025_035 = R_Filters_025_035["G_Detect_ID"]
R_Z_025_035 = R_Filters_025_035["G_Z"]

I_Data_025_035 = I_Filters_025_035["G_Data"]
I_RA_025_035 = I_Filters_025_035["G_RA"]
I_DEC_025_035 = I_Filters_025_035["G_DEC"]
I_Detect_ID_025_035 = I_Filters_025_035["G_Detect_ID"]
I_Z_025_035 = I_Filters_025_035["G_Z"]

Z_Data_025_035 = Z_Filters_025_035["G_Data"]
Z_RA_025_035 = Z_Filters_025_035["G_RA"]
Z_DEC_025_035 = Z_Filters_025_035["G_DEC"]
Z_Detect_ID_025_035 = Z_Filters_025_035["G_Detect_ID"]
Z_Z_025_035 = Z_Filters_025_035["G_Z"]

Y_Data_025_035 = Y_Filters_025_035["G_Data"]
Y_RA_025_035 = Y_Filters_025_035["G_RA"]
Y_DEC_025_035 = Y_Filters_025_035["G_DEC"]
Y_Detect_ID_025_035 = Y_Filters_025_035["G_Detect_ID"]
Y_Z_025_035 = Y_Filters_025_035["G_Z"]

In [ ]:
G_Data_035_045 = G_Filters_035_045["G_Data"]
G_RA_035_045 = G_Filters_035_045["G_RA"]
G_DEC_035_045 = G_Filters_035_045["G_DEC"]
G_Detect_ID_035_045 = G_Filters_035_045["G_Detect_ID"]
G_Z_035_045 = G_Filters_035_045["G_Z"]

R_Data_035_045 = R_Filters_035_045["G_Data"]
R_RA_035_045 = R_Filters_035_045["G_RA"]
R_DEC_035_045 = R_Filters_035_045["G_DEC"]
R_Detect_ID_035_045 = R_Filters_035_045["G_Detect_ID"]
R_Z_035_045 = R_Filters_035_045["G_Z"]

I_Data_035_045 = I_Filters_035_045["G_Data"]
I_RA_035_045 = I_Filters_035_045["G_RA"]
I_DEC_035_045 = I_Filters_035_045["G_DEC"]
I_Detect_ID_035_045 = I_Filters_035_045["G_Detect_ID"]
I_Z_035_045 = I_Filters_035_045["G_Z"]

Z_Data_035_045 = Z_Filters_035_045["G_Data"]
Z_RA_035_045 = Z_Filters_035_045["G_RA"]
Z_DEC_035_045 = Z_Filters_035_045["G_DEC"]
Z_Detect_ID_035_045 = Z_Filters_035_045["G_Detect_ID"]
Z_Z_035_045 = Z_Filters_035_045["G_Z"]

Y_Data_035_045 = Y_Filters_035_045["G_Data"]
Y_RA_035_045 = Y_Filters_035_045["G_RA"]
Y_DEC_035_045 = Y_Filters_035_045["G_DEC"]
Y_Detect_ID_035_045 = Y_Filters_035_045["G_Detect_ID"]
Y_Z_035_045 = Y_Filters_035_045["G_Z"]

In [ ]:
G_Data_045_055 = G_Filters_045_055["G_Data"]
G_RA_045_055 = G_Filters_045_055["G_RA"]
G_DEC_045_055 = G_Filters_045_055["G_DEC"]
G_Detect_ID_045_055 = G_Filters_045_055["G_Detect_ID"]
G_Z_045_055 = G_Filters_045_055["G_Z"]

R_Data_045_055 = R_Filters_045_055["G_Data"]
R_RA_045_055 = R_Filters_045_055["G_RA"]
R_DEC_045_055 = R_Filters_045_055["G_DEC"]
R_Detect_ID_045_055 = R_Filters_045_055["G_Detect_ID"]
R_Z_045_055 = R_Filters_045_055["G_Z"]

I_Data_045_055 = I_Filters_045_055["G_Data"]
I_RA_045_055 = I_Filters_045_055["G_RA"]
I_DEC_045_055 = I_Filters_045_055["G_DEC"]
I_Detect_ID_045_055 = I_Filters_045_055["G_Detect_ID"]
I_Z_045_055 = I_Filters_045_055["G_Z"]

Z_Data_045_055 = Z_Filters_045_055["G_Data"]
Z_RA_045_055 = Z_Filters_045_055["G_RA"]
Z_DEC_045_055 = Z_Filters_045_055["G_DEC"]
Z_Detect_ID_045_055 = Z_Filters_045_055["G_Detect_ID"]
Z_Z_045_055 = Z_Filters_045_055["G_Z"]

Y_Data_045_055 = Y_Filters_045_055["G_Data"]
Y_RA_045_055 = Y_Filters_045_055["G_RA"]
Y_DEC_045_055 = Y_Filters_045_055["G_DEC"]
Y_Detect_ID_045_055 = Y_Filters_045_055["G_Detect_ID"]
Y_Z_045_055 = Y_Filters_045_055["G_Z"]

In [ ]:
G_Data_055_065 = G_Filters_055_065["G_Data"]
G_RA_055_065 = G_Filters_055_065["G_RA"]
G_DEC_055_065 = G_Filters_055_065["G_DEC"]
G_Detect_ID_055_065 = G_Filters_055_065["G_Detect_ID"]
G_Z_055_065 = G_Filters_055_065["G_Z"]

R_Data_055_065 = R_Filters_055_065["G_Data"]
R_RA_055_065 = R_Filters_055_065["G_RA"]
R_DEC_055_065 = R_Filters_055_065["G_DEC"]
R_Detect_ID_055_065 = R_Filters_055_065["G_Detect_ID"]
R_Z_055_065 = R_Filters_055_065["G_Z"]

I_Data_055_065 = I_Filters_055_065["G_Data"]
I_RA_055_065 = I_Filters_055_065["G_RA"]
I_DEC_055_065 = I_Filters_055_065["G_DEC"]
I_Detect_ID_055_065 = I_Filters_055_065["G_Detect_ID"]
I_Z_055_065 = I_Filters_055_065["G_Z"]

Z_Data_055_065 = Z_Filters_055_065["G_Data"]
Z_RA_055_065 = Z_Filters_055_065["G_RA"]
Z_DEC_055_065 = Z_Filters_055_065["G_DEC"]
Z_Detect_ID_055_065 = Z_Filters_055_065["G_Detect_ID"]
Z_Z_055_065 = Z_Filters_055_065["G_Z"]

Y_Data_055_065 = Y_Filters_055_065["G_Data"]
Y_RA_055_065 = Y_Filters_055_065["G_RA"]
Y_DEC_055_065 = Y_Filters_055_065["G_DEC"]
Y_Detect_ID_055_065 = Y_Filters_055_065["G_Detect_ID"]
Y_Z_055_065 = Y_Filters_055_065["G_Z"]

In [ ]:
G_Data_065_075 = G_Filters_065_075["G_Data"]
G_RA_065_075 = G_Filters_065_075["G_RA"]
G_DEC_065_075 = G_Filters_065_075["G_DEC"]
G_Detect_ID_065_075 = G_Filters_065_075["G_Detect_ID"]
G_Z_065_075 = G_Filters_065_075["G_Z"]

R_Data_065_075 = R_Filters_065_075["G_Data"]
R_RA_065_075 = R_Filters_065_075["G_RA"]
R_DEC_065_075 = R_Filters_065_075["G_DEC"]
R_Detect_ID_065_075 = R_Filters_065_075["G_Detect_ID"]
R_Z_065_075 = R_Filters_065_075["G_Z"]

I_Data_065_075 = I_Filters_065_075["G_Data"]
I_RA_065_075 = I_Filters_065_075["G_RA"]
I_DEC_065_075 = I_Filters_065_075["G_DEC"]
I_Detect_ID_065_075 = I_Filters_065_075["G_Detect_ID"]
I_Z_065_075 = I_Filters_065_075["G_Z"]

Z_Data_065_075 = Z_Filters_065_075["G_Data"]
Z_RA_065_075 = Z_Filters_065_075["G_RA"]
Z_DEC_065_075 = Z_Filters_065_075["G_DEC"]
Z_Detect_ID_065_075 = Z_Filters_065_075["G_Detect_ID"]
Z_Z_065_075 = Z_Filters_065_075["G_Z"]

Y_Data_065_075 = Y_Filters_065_075["G_Data"]
Y_RA_065_075 = Y_Filters_065_075["G_RA"]
Y_DEC_065_075 = Y_Filters_065_075["G_DEC"]
Y_Detect_ID_065_075 = Y_Filters_065_075["G_Detect_ID"]
Y_Z_065_075 = Y_Filters_065_075["G_Z"]

In [ ]:
G_Data_075_085 = G_Filters_075_085["G_Data"]
G_RA_075_085 = G_Filters_075_085["G_RA"]
G_DEC_075_085 = G_Filters_075_085["G_DEC"]
G_Detect_ID_075_085 = G_Filters_075_085["G_Detect_ID"]
G_Z_075_085 = G_Filters_075_085["G_Z"]

R_Data_075_085 = R_Filters_075_085["G_Data"]
R_RA_075_085 = R_Filters_075_085["G_RA"]
R_DEC_075_085 = R_Filters_075_085["G_DEC"]
R_Detect_ID_075_085 = R_Filters_075_085["G_Detect_ID"]
R_Z_075_085 = R_Filters_075_085["G_Z"]

I_Data_075_085 = I_Filters_075_085["G_Data"]
I_RA_075_085 = I_Filters_075_085["G_RA"]
I_DEC_075_085 = I_Filters_075_085["G_DEC"]
I_Detect_ID_075_085 = I_Filters_075_085["G_Detect_ID"]
I_Z_075_085 = I_Filters_075_085["G_Z"]

Z_Data_075_085 = Z_Filters_075_085["G_Data"]
Z_RA_075_085 = Z_Filters_075_085["G_RA"]
Z_DEC_075_085 = Z_Filters_075_085["G_DEC"]
Z_Detect_ID_075_085 = Z_Filters_075_085["G_Detect_ID"]
Z_Z_075_085 = Z_Filters_075_085["G_Z"]

Y_Data_075_085 = Y_Filters_075_085["G_Data"]
Y_RA_075_085 = Y_Filters_075_085["G_RA"]
Y_DEC_075_085 = Y_Filters_075_085["G_DEC"]
Y_Detect_ID_075_085 = Y_Filters_075_085["G_Detect_ID"]
Y_Z_075_085 = Y_Filters_075_085["G_Z"]

In [ ]:
G_Data_085_096 = G_Filters_085_096["G_Data"]
G_RA_085_096 = G_Filters_085_096["G_RA"]
G_DEC_085_096 = G_Filters_085_096["G_DEC"]
G_Detect_ID_085_096 = G_Filters_085_096["G_Detect_ID"]
G_Z_085_096 = G_Filters_085_096["G_Z"]

R_Data_085_096 = R_Filters_085_096["G_Data"]
R_RA_085_096 = R_Filters_085_096["G_RA"]
R_DEC_085_096 = R_Filters_085_096["G_DEC"]
R_Detect_ID_085_096 = R_Filters_085_096["G_Detect_ID"]
R_Z_085_096 = R_Filters_085_096["G_Z"]

I_Data_085_096 = I_Filters_085_096["G_Data"]
I_RA_085_096 = I_Filters_085_096["G_RA"]
I_DEC_085_096 = I_Filters_085_096["G_DEC"]
I_Detect_ID_085_096 = I_Filters_085_096["G_Detect_ID"]
I_Z_085_096 = I_Filters_085_096["G_Z"]

Z_Data_085_096 = Z_Filters_085_096["G_Data"]
Z_RA_085_096 = Z_Filters_085_096["G_RA"]
Z_DEC_085_096 = Z_Filters_085_096["G_DEC"]
Z_Detect_ID_085_096 = Z_Filters_085_096["G_Detect_ID"]
Z_Z_085_096 = Z_Filters_085_096["G_Z"]

Y_Data_085_096 = Y_Filters_085_096["G_Data"]
Y_RA_085_096 = Y_Filters_085_096["G_RA"]
Y_DEC_085_096 = Y_Filters_085_096["G_DEC"]
Y_Detect_ID_085_096 = Y_Filters_085_096["G_Detect_ID"]
Y_Z_085_096 = Y_Filters_085_096["G_Z"]

In [ ]:
fig = plt.figure(figsize=(10,10))

ax = fig.add_subplot(1,1, 1)

im = ax.imshow(Y_Data_085_096[0], origin="lower", cmap="YlOrBr")

#plt.scatter(X, Y, color="skyblue", s=200, marker="+")

#sep_ap = cutout['details']['sep_objects'][cutout['details']['sep_obj_idx']]

coord_offset = 0.5 * (ax.get_xlim()[0] + ax.get_xlim()[1])

plt.colorbar(im, ax=ax)

#filename = str(Data_Confirmed_Redshift_And_AGN_Status["Detectid"][200])

#plt.title(str(filename))

In [ ]:
3